# Indian Pines — benchmark chính thức bốn quy trình huấn luyện

**CNN thường · CNN + Random Oversampling (ROS) · CNN + trọng số lớp · 3D-HyperGAMO**

Notebook này là **benchmark nền chính thức** của khóa luận trên giao thức DIP 16 lớp.

Quy trình: **đọc dữ liệu → kiểm tra mất cân bằng → chia train/validation/test → fit PCA/MinMax trên train → cắt patch → smoke test → huấn luyện/tuning bằng validation → lưu checkpoint → so sánh validation → chỉ mở test khi toàn bộ cấu hình cuối đã khóa.**

Notebook giữ đủ **16 lớp**, patch **11 × 11**, 40 thành phần PCA và cùng train/validation/test cho cả bốn phương pháp. Giao thức DIP hiện tại vẫn có chồng lấn patch; **không gọi đây là spatial leakage-free benchmark**. Thí nghiệm spatial-controlled là một experiment riêng.

Ba CNN dùng đúng kiến trúc classifier của 3D-HyperGAMO. CNN thường và Class Weight không sinh mẫu; ROS lặp mẫu thật; 3D-HyperGAMO tạo mẫu mới.


## 1. Cách chạy — cấu hình benchmark đã khóa

1. Gắn dataset có `Indian_pines_corrected.mat` và `Indian_pines_gt.mat` vào Input.
2. Bật GPU T4; notebook dùng **một GPU** và chạy các phương pháp lần lượt.
3. Giữ nguyên cấu hình benchmark bên dưới. Chọn **Save Version → Save & Run All**.
4. Notebook checkpoint liên tục. Nếu hết phiên, dùng ZIP do **chính notebook này** tạo để resume với cấu hình không đổi.

### Hợp đồng benchmark

- Seed: **42, 43, 44** cho cả bốn phương pháp.
- Batch size của ba CNN cố định: **64**.
- Learning rate của ba CNN được chọn từ cùng search space: **1e-3, 5e-4, 2e-4, 1e-4**.
- Checkpoint được chọn bằng **Macro-F1 validation**.
- Không dùng test để chọn learning rate, epoch, GAN hay cải tiến.
- Search space không mở rộng sau khi xem kết quả chỉ để làm số đẹp hơn.
- Chỉ sửa benchmark nếu phát hiện bug hoặc protocol sai.

Mặc định `EVALUATE_TEST=False`. Khi toàn bộ cấu hình GAN cuối/phương pháp cải tiến đã khóa, bật `True` để đánh giá test. Các baseline đã train sẽ không bị train lại nếu resume đúng ZIP/run.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, platform, subprocess, importlib.metadata, shutil, zipfile

CFG = dict(protocol='dip_train_only_validation', accepted_class_ids=list(range(1,17)),
    patch_size=11, pca_components=40, pca_solver='full', split_seed=42,
    validation_fraction=0.20, train_seed=42, batch_size=100, max_step=8000,
    eval_every=500, eval_batch_size=32, max_minutes=180, gpu_index=0, resume_training=True)
COMPARISON = dict(
    seeds=[42,43,44],
    methods=['cnn3d_real_only','cnn3d_random_oversampling','cnn3d_class_weighting'],
    learning_rates=[0.001,0.0005,0.0002,0.0001],
    batch_size=32,
    max_epochs=150,
    early_patience=18,
    lr_patience=6,
    min_lr=0.000001,
)
RUN_TRAINING = True
EVALUATE_TEST = False
INSTALL_COMPATIBILITY_PACKAGES = True
HSI_PATH = None  # điền đường dẫn nếu Input có nhiều bản sao cùng tên
GT_PATH = None
RESUME_ZIP = None  # ví dụ '/kaggle/input/my-results/comparison_4methods_....zip'
LOCAL_DATA = Path(r'D:\HUIT\Chương trình học\HK1 ( 2026- 2027)\KLTN - GAN\dataset')
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'four_method_outputs'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
assert CFG['protocol']=='dip_train_only_validation' and CFG['accepted_class_ids']==list(range(1,17))
assert len(COMPARISON['seeds'])==len(set(COMPARISON['seeds']))==3
assert CFG['max_step']%2==0 and CFG['max_step']>0
print('Các seed benchmark:', COMPARISON['seeds'])
print('GAN:', CFG['max_step'], 'lượt cập nhật bộ phân lớp; CNN tối đa', COMPARISON['max_epochs'],'epoch / cấu hình')
print('CNN LR search:', COMPARISON['learning_rates'], '| batch:', COMPARISON['batch_size'])
print('EVALUATE_TEST:', EVALUATE_TEST)
print('Đầu ra:', OUTPUT_ROOT)


## 2. Kiểm tra môi trường

TensorFlow chỉ chạy trong tiến trình con. Mỗi phương pháp kết thúc sẽ giải phóng GPU trước phương pháp tiếp theo.

In [ ]:
# Không import TensorFlow trong tiến trình notebook.
required = {'tensorflow': '2.20.0', 'tf-keras': '2.20.0'}
missing = []
for package, wanted in required.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    print(package, installed, '→ yêu cầu', wanted)
    if installed != wanted:
        missing.append(f'{package}=={wanted}')
if missing:
    if not INSTALL_COMPATIBILITY_PACKAGES:
        raise RuntimeError('Cần cài gói tương thích: ' + ' '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True)
for package, wanted in required.items():
    assert importlib.metadata.version(package) == wanted

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.io import loadmat
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import joblib
from IPython.display import display, FileLink
print('Python:', platform.python_version(), '| NumPy:', np.__version__)


In [ ]:
if RUN_TRAINING:
    subprocess.run([sys.executable,'-c',
        "import tensorflow as tf; g=tf.config.list_physical_devices('GPU'); print('GPU:',g); assert g, 'Cần bật GPU trước khi chạy'"],check=True)
plt.rcParams.update({'figure.dpi':110,'axes.titlesize':11,'font.size':10})
from IPython.display import Markdown


## 3. Các hàm nền và mã gốc

Ô mã dài dưới đây chứa nguồn mạng của tác giả và bộ chạy đã được đóng gói. Không cần sửa ô này. Các ô tiếp theo vẫn tách từng thao tác và hiển thị kết quả ngay tại chỗ.

In [ ]:
DATA_SOURCE = '"""Fixed author DIP masks: retain all 16 classes and report, never hide, overlap."""\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\nfrom scipy.ndimage import maximum_filter\n\nCLASS_NAMES = [\'Alfalfa\', \'Corn-notill\', \'Corn-mintill\', \'Corn\', \'Grass-pasture\',\n    \'Grass-trees\', \'Grass-pasture-mowed\', \'Hay-windrowed\', \'Oats\', \'Soybean-notill\',\n    \'Soybean-mintill\', \'Soybean-clean\', \'Wheat\', \'Woods\',\n    \'Buildings-Grass-Trees-Drives\', \'Stone-Steel-Towers\']\nDIP_MAPPING = np.array([0, 2, 3, 5, 6, 8, 10, 11, 12, 14, 1, 4, 7, 9, 13, 15, 16])\nEXPECTED_TRAIN = np.array([29, 762, 435, 146, 232, 394, 16, 235, 10, 470, 1424, 328, 132, 728, 291, 57])\nEXPECTED_TEST = np.array([25, 675, 404, 99, 274, 354, 2, 250, 10, 503, 1065, 282, 80, 545, 99, 44])\n\n\ndef digest_file(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef dip_split(gt, encoded_mask, cfg):\n    assert encoded_mask.shape == gt.shape == (145, 145)\n    assert np.issubdtype(encoded_mask.dtype, np.integer)\n    assert encoded_mask.min() >= 0 and encoded_mask.max() <= 16\n    author_train = DIP_MAPPING[encoded_mask].astype(\'int16\')\n    test_labels = np.where(author_train == 0, gt, 0).astype(\'int16\')\n    assert np.array_equal([(author_train == c).sum() for c in range(1, 17)], EXPECTED_TRAIN)\n    assert np.array_equal([(test_labels == c).sum() for c in range(1, 17)], EXPECTED_TEST)\n    mismatch = (author_train > 0) & (author_train != gt)\n    label_audit = dict(train_vs_standard_gt_disagreements=int(mismatch.sum()),\n        assigned_on_gt_background=int(((author_train > 0) & (gt == 0)).sum()),\n        assigned_different_nonzero_label=int((mismatch & (gt > 0)).sum()))\n    # Reproduction keeps author train labels, rather than silently changing 161 samples.\n    labels = np.where(author_train > 0, author_train, test_labels)\n    split = np.full(gt.shape, -1, dtype=\'int8\')\n    split[author_train > 0] = 0\n    split[test_labels > 0] = 2\n    if cfg[\'protocol\'] == \'dip_train_only_validation\':\n        rng = np.random.default_rng(cfg[\'split_seed\'])\n        for cid in range(1, 17):\n            indices = np.flatnonzero(author_train.ravel() == cid)\n            n_val = max(2, int(np.floor(len(indices) * cfg[\'validation_fraction\'])))\n            if n_val > len(indices) - 2:\n                raise ValueError(\'Không đủ train/validation cho lớp \' + str(cid))\n            split.ravel()[rng.permutation(indices)[:n_val]] = 1\n    elif cfg[\'protocol\'] != \'dip_reproduction\':\n        raise ValueError(\'Protocol không hợp lệ.\')\n    coords = [np.argwhere(split == i) for i in range(3)]\n    return split, labels, coords, label_audit\n\n\ndef audit_overlap(split, patch_size):\n    centers = [split == i for i in range(3)]\n    footprints = [maximum_filter(m.astype(\'uint8\'), size=patch_size, mode=\'constant\') > 0 for m in centers]\n    report = {}\n    names = [\'train\', \'validation\', \'test\']\n    for a, b in [(0, 1), (0, 2), (1, 2)]:\n        prefix = names[a] + \'_\' + names[b]\n        report[prefix + \'_shared_centers\'] = int((centers[a] & centers[b]).sum())\n        report[prefix + \'_shared_raw_pixels\'] = int((footprints[a] & footprints[b]).sum())\n        # A patch intersects another patch when center distance <= 2 * radius.\n        near_a = maximum_filter(centers[a].astype(\'uint8\'), size=2 * patch_size - 1, mode=\'constant\') > 0\n        report[prefix + \'_overlapping_centers_in_second_set\'] = int((near_a & centers[b]).sum())\n    assert all(v == 0 for k, v in report.items() if k.endswith(\'_shared_centers\'))\n    # Nonzero RAW footprint overlap is expected here; it is recorded, not asserted away.\n    return report\n\n\ndef extract_patches(cube, coords, patch_size):\n    if patch_size <= 0 or patch_size % 2 != 1:\n        raise ValueError(\'patch_size phải lẻ, dương.\')\n    coords = np.asarray(coords)\n    if coords.ndim != 2 or coords.shape[1] != 2 or not np.issubdtype(coords.dtype, np.integer):\n        raise ValueError(\'Tọa độ cần có dạng (N, 2), số nguyên.\')\n    if len(coords) and ((coords < 0).any() or (coords >= np.array(cube.shape[:2])).any()):\n        raise ValueError(\'Tọa độ ngoài ảnh.\')\n    r = patch_size // 2\n    padded = np.pad(cube, ((r, r), (r, r), (0, 0)), mode=\'constant\')\n    out = np.empty((len(coords), patch_size, patch_size, cube.shape[-1], 1), dtype=\'float32\')\n    for i, (row, col) in enumerate(coords):\n        out[i, ..., 0] = padded[row:row+patch_size, col:col+patch_size]\n    if not np.isfinite(out).all():\n        raise ValueError(\'Dữ liệu có NaN/Inf.\')\n    return out\n'
exec(compile(DATA_SOURCE, 'dip_data.py', 'exec'))


In [ ]:
AUTHOR_NETWORKS = r'''
# For running in python 2.x
from __future__ import print_function, unicode_literals
from __future__ import absolute_import, division
import numpy as np
import tensorflow as tf
from keras import backend as K
from keras.layers import Input, Dropout, Dense, RepeatVector, Lambda, Reshape, Conv3D, Conv2D, Flatten, InputSpec
from keras.layers import BatchNormalization, Concatenate, Multiply, Add, Conv2DTranspose, GlobalAveragePooling2D
from keras.layers.advanced_activations import LeakyReLU, Softmax
from keras.models import Model

def denseGamoGenCreate(latDim, num_class):
    noise = Input(shape=(latDim, ))
    labels = Input(shape=(num_class, ))
    gamoGenInput = Concatenate()([noise, labels])

    x = Dense(7 * 7 * 256, use_bias=False)(gamoGenInput)
    x = BatchNormalization(momentum=0.9)(x)
    x = LeakyReLU()(x)

    x = Reshape((7, 7, 256))(x)

    x = Conv2DTranspose(
        128, (5, 5), strides=(1, 1), padding='same', use_bias=False)(x)
    x = BatchNormalization(momentum=0.9)(x)
    x = LeakyReLU()(x)

    x = Conv2DTranspose(
        64, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = BatchNormalization(momentum=0.9)(x)
    x = LeakyReLU()(x)

    x = Conv2DTranspose(
        1, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = BatchNormalization(momentum=0.9)(x)

    gamoGenFinal = Flatten()(x)

    gamoGen = Model([noise, labels], gamoGenFinal)
    return gamoGen



def denseGenProcessCreate(numMinor, dataMinor,sh,mul):
    ip1=Input(shape=(784,))
    x=Dense(numMinor, activation='softmax')(ip1)
    x=RepeatVector(mul)(x)
    z = np.reshape(dataMinor,(numMinor,mul))
    genProcessFinal=Lambda(lambda x: K.sum(x*K.transpose(K.constant(z)), axis=2))(x)
    genProcessReshape = Reshape(sh)(genProcessFinal)
    genProcess=Model(ip1, genProcessReshape)
    return genProcess

def denseDisCreate(sh, num_class):
    imIn=Input(shape=sh)
    conv_layer1 = Conv3D(filters=8, kernel_size=(1, 1, 7), activation='relu', padding='same')(imIn)
    conv_layer2 = Conv3D(filters=16, kernel_size=(3, 3, 5), activation='relu',padding='same')(conv_layer1)
    conv_layer3 = Conv3D(filters=32, kernel_size=(5, 5, 7), activation='relu',padding='same')(conv_layer2)
    conv3d_shape = conv_layer3.shape
    conv_layer3 = Reshape((conv3d_shape[1], conv3d_shape[2], conv3d_shape[3]*conv3d_shape[4]))(conv_layer3)
    conv_layer4 = Conv2D(filters=64, kernel_size=(3,3), activation='relu',padding='same')(conv_layer3)
    conv_layer4 = GlobalAveragePooling2D()(conv_layer4)
    flatten_layer = Flatten()(conv_layer4)
    labels=Input(shape=(num_class,))
    disInput=Concatenate()([flatten_layer, labels])
    x=Dropout(0.5)(disInput)
    disFinal=Dense(1, activation='sigmoid', kernel_initializer="he_normal")(x)
    dis=Model([imIn, labels], disFinal)
    return dis

def denseMlpCreate(sh, num_class):
    imIn=Input(shape=sh)
    conv_layer1 = Conv3D(filters=8, kernel_size=(1, 1, 7), activation='relu', padding='same')(imIn)
    conv_layer2 = Conv3D(filters=16, kernel_size=(3, 3, 5), activation='relu',padding='same')(conv_layer1)
    conv_layer3 = Conv3D(filters=32, kernel_size=(5, 5, 7), activation='relu',padding='same')(conv_layer2)
    conv3d_shape = conv_layer3.shape
    conv_layer3 = Reshape((conv3d_shape[1], conv3d_shape[2], conv3d_shape[3]*conv3d_shape[4]))(conv_layer3)
    conv_layer4 = Conv2D(filters=64, kernel_size=(3,3), activation='relu',padding='same')(conv_layer3)
    conv_layer4 = GlobalAveragePooling2D()(conv_layer4)
    flatten_layer = Flatten()(conv_layer4)
    x=Dropout(0.5)(flatten_layer)
    mlpFinal = Dense(num_class, activation="softmax", kernel_initializer="he_normal")(x)
    mlp=Model(imIn, mlpFinal)
    return mlp

'''
print('Đã nạp mã gốc: dense_net3D_SA.py')


In [ ]:
AUTHOR_MAIN = r'''
import os
import auxil
import argparse
import numpy as np
import dense_suppli as spp
import dense_net3D_SA as nt
from keras.optimizers import Adam
from keras.utils.np_utils import to_categorical
from keras.layers import Input
from keras.models import Model



def load_hyper(args):
    if args.dataset not in ["UH", "DIP", "DUP"]:
        acrstate = None if args.random_state == None else args.random_state+args.idtest
        pixelsO, labelsO, numclass = auxil.loadData(args.dataset, num_components=args.components, rand_state=acrstate)
        pixelsO, labelsO = auxil.createImageCubes(pixelsO, labelsO, windowSize=args.spatialsize, removeZeroLabels = False)
        pixels = pixelsO[labelsO!=0]
        labels = labelsO[labelsO!=0] - 1
        bands = pixels.shape[-1]; numberofclass = len(np.unique(labels))
        pixels = pixels.reshape((pixels.shape[0], args.spatialsize, args.spatialsize, bands, 1))
        pixelsO = pixelsO.reshape((pixelsO.shape[0], args.spatialsize, args.spatialsize, bands, 1))
        x_train, x_test, y_train, y_test = auxil.split_data(pixels, labels, args.tr_percent, rand_state=acrstate)
        del pixels, labels
    else:
        x_train, x_test, y_train, y_test, pixelsO, labelsO, bands, numberofclass = auxil.loadDataFIX(args)
    return (x_train, y_train), (x_test, y_test), (pixelsO, labelsO), numberofclass, bands





def main():
    parser = argparse.ArgumentParser(description='PyTorch DCNNs Training')
    parser.add_argument('--max_step', default=8000, type=int, help='number of total epochs to run')
    parser.add_argument('--idtest', default=0, type=int, help='id of experiment')
    parser.add_argument('--lr', '--learning-rate', default=0.1, type=float, help='initial learning rate')
    parser.add_argument('--components', default=None, type=int, help='dimensionality reduction')
    parser.add_argument('--dataset', default='IP', type=str, help='dataset (options: IP, UP, SV, KSC)')
    parser.add_argument('--tr_percent', default=0.10, type=float, help='samples of train set')
    parser.add_argument('--tr_bsize', default=100, type=int, help='mini-batch train size (default: 100)')
    parser.add_argument('--inplanes', dest='inplanes', default=16, type=int, help='bands before blocks')
    parser.add_argument('--no-bottleneck', dest='bottleneck', action='store_false', help='to use basicblock (default: bottleneck)')
    parser.add_argument('--spatialsize', dest='spatialsize', default=7, type=int, help='spatial-spectral patch dimension')
    parser.add_argument('--momentum', default=0.9, type=float, help='momentum')
    parser.add_argument('--weight-decay', '--wd', default=1e-4, type=float, help='weight decay (default: 1e-4)')
    parser.add_argument('--random_state', default=None, type=int, help='random seed')

    args = parser.parse_args()
    state = {k: v for k, v in args._get_kwargs()}

    #args.components = 35 if args.dataset == 'IP' else 15
    (trainS, labelTr), (testS, labelTs), (pixels, labels), numberofclass, bands = load_hyper(args)


    n, m = trainS.shape[0], testS.shape[0]
    labelTr, labelTs, c, pInClass, _, newOrder = spp.relabel(labelTr, labelTs)
    imbalancedCls, toBalance, imbClsNum, ir=spp.irFind(pInClass, c)

    labelsCat = to_categorical(labelTr)

    shuffleIndex=np.random.choice(np.arange(n), size=(n,), replace=False)
    trainS=trainS[shuffleIndex]
    labelTr=labelTr[shuffleIndex]
    labelsCat=labelsCat[shuffleIndex]
    classMap=list()
    for i in range(c):
        classMap.append(np.where(labelTr==i)[0])


    adamOpt=Adam(0.0002, 0.5)
    latDim, modelSamplePd, resSamplePd=100, 8000, 500

    # model initialization
    mlp=nt.denseMlpCreate(sh=(args.spatialsize, args.spatialsize, bands, 1, ), num_class=numberofclass)
    mlp.compile(loss='mean_squared_error', optimizer=adamOpt)
    mlp.trainable=False

    dis=nt.denseDisCreate((args.spatialsize, args.spatialsize, bands, 1, ), num_class=numberofclass)
    dis.compile(loss='mean_squared_error', optimizer=adamOpt)
    dis.trainable=False

    gen=nt.denseGamoGenCreate(latDim, numberofclass)


    gen_processed, genP_mlp, genP_dis=list(), list(), list()
    for i in range(imbClsNum):
        dataMinor=trainS[classMap[i], :]
        numMinor=dataMinor.shape[0]
        gen_processed.append(nt.denseGenProcessCreate(numMinor, dataMinor,sh = \
            (args.spatialsize, args.spatialsize, bands, 1),mul = args.spatialsize * args.spatialsize * bands))

        ip1=Input(shape=(latDim,))
        ip2=Input(shape=(c,))
        op1=gen([ip1, ip2])
        op2=gen_processed[i](op1)
        op3=mlp(op2)
        genP_mlp.append(Model(inputs=[ip1, ip2], outputs=op3))
        genP_mlp[i].compile(loss='mean_squared_error', optimizer=adamOpt)

        ip1=Input(shape=(latDim,))
        ip2=Input(shape=(c,))
        ip3=Input(shape=(c,))
        op1=gen([ip1, ip2])
        op2=gen_processed[i](op1)
        op3=dis([op2, ip3])
        genP_dis.append(Model(inputs=[ip1, ip2, ip3], outputs=op3))
        genP_dis[i].compile(loss='mean_squared_error', optimizer=adamOpt)


    batchDiv, numBatches, bSStore = spp.batchDivision(n, args.tr_bsize)
    genClassPoints=int(np.ceil(args.tr_bsize / c))

    fileStart = './SavedModel/'
    savePath = './SavedModel/'
    fileEnd = '_Model.h5'
    if not os.path.exists(fileStart):
        os.makedirs(fileStart)
    picPath=savePath+'Pictures'
    if not os.path.exists(picPath):
        os.makedirs(picPath)


    iter=np.int(np.ceil(args.max_step/resSamplePd)+1)
    acsaSaveTr, gmSaveTr, accSaveTr=np.zeros((iter,)), np.zeros((iter,)), np.zeros((iter,))
    acsaSaveTs, gmSaveTs, accSaveTs=np.zeros((iter,)), np.zeros((iter,)), np.zeros((iter,))
    confMatSaveTr, confMatSaveTs=np.zeros((iter, c, c)), np.zeros((iter, c, c))
    tprSaveTr, tprSaveTs=np.zeros((iter, c)), np.zeros((iter, c))



    step=0
    bestacc = -1
    while step < args.max_step:
        for j in range(numBatches):
            x1, x2=batchDiv[j, 0], batchDiv[j+1, 0]
            validR=np.ones((bSStore[j, 0],1))-np.random.uniform(0,0.1, size=(bSStore[j, 0], 1))
            mlp.train_on_batch(trainS[x1:x2], labelsCat[x1:x2])
            dis.train_on_batch([trainS[x1:x2], labelsCat[x1:x2]], validR)

            invalid=np.zeros((bSStore[j, 0], 1))+np.random.uniform(0, 0.1, size=(bSStore[j, 0], 1))
            randNoise=np.random.normal(0, 1, (bSStore[j, 0], latDim))
            fakeLabel=spp.randomLabelGen(toBalance, bSStore[j, 0], c)
            rLPerClass=spp.rearrange(fakeLabel, imbClsNum)
            fakePoints=np.zeros((bSStore[j, 0],args.spatialsize, args.spatialsize, bands, 1))
            genFinal=gen.predict([randNoise, fakeLabel])
            for i1 in range(imbClsNum):
                if rLPerClass[i1].shape[0]!=0:
                    temp=genFinal[rLPerClass[i1]]
                    fakePoints[rLPerClass[i1]]=gen_processed[i1].predict(temp)

            mlp.train_on_batch(fakePoints, fakeLabel)
            dis.train_on_batch([fakePoints, fakeLabel], invalid)

            for i1 in range(imbClsNum):
                validA=np.ones((genClassPoints, 1))
                randomLabel=np.zeros((genClassPoints, c))
                randomLabel[:, i1]=1
                randNoise=np.random.normal(0, 1, (genClassPoints, latDim))
                oppositeLabel=np.ones((genClassPoints, c))-randomLabel
                genP_mlp[i1].train_on_batch([randNoise, randomLabel], oppositeLabel)
                genP_dis[i1].train_on_batch([randNoise, randomLabel, randomLabel], validA)

            if step%resSamplePd==0:
                saveStep=int(step//resSamplePd)

                pLabel=np.argmax(mlp.predict(trainS), axis=1)
                acsa, gm, tpr, confMat, acc=spp.indices(pLabel, labelTr)
                print('Train: Step: ', step, 'ACSA: ', np.round(acsa, 4), 'GM: ', np.round(gm, 4))
                print('TPR: ', np.round(tpr, 2))
                acsaSaveTr[saveStep], gmSaveTr[saveStep], accSaveTr[saveStep]=acsa, gm, acc
                confMatSaveTr[saveStep]=confMat
                tprSaveTr[saveStep]=tpr

                pLabel=np.argmax(mlp.predict(testS), axis=1)
                acsa, gm, tpr, confMat, acc=spp.indices(pLabel, labelTs)
                print('Test: Step: ', step, 'ACSA: ', np.round(acsa, 4), 'GM: ', np.round(gm, 4))
                print('TPR: ', np.round(tpr, 2))
                acsaSaveTs[saveStep], gmSaveTs[saveStep], accSaveTs[saveStep]=acsa, gm, acc
                confMatSaveTs[saveStep]=confMat
                tprSaveTs[saveStep]=tpr

                results = auxil.reports(pLabel, labelTs)[2]
                if bestacc <= results[0]:
                    bestacc = auxil.reports(pLabel, labelTs)[2][0]
                    resultsF = auxil.reports(pLabel, labelTs)[2]

            if step%modelSamplePd==0 and step!=0:
                direcPath=savePath+'gamo_models_'+str(step)
                if not os.path.exists(direcPath):
                    os.makedirs(direcPath)
                gen.save(direcPath+'/GEN_'+str(step)+fileEnd)
                mlp.save(direcPath+'/MLP_'+str(step)+fileEnd)
                dis.save(direcPath+'/DIS_'+str(step)+fileEnd)
                for i in range(imbClsNum):
                    gen_processed[i].save(direcPath+'/GenP_'+str(i)+'_'+str(step)+fileEnd)

            step=step+2
            if step>=args.max_step: break

    print(newOrder)
    print(resultsF)

if __name__ == '__main__':
	main()

'''
print('Đã nạp mã gốc: main.py')


In [ ]:
AUTHOR_REFERENCE_FILES = {'auxil.py': 'import os\nimport numpy as np\nimport scipy.io as sio\nfrom sklearn.decomposition import PCA\nfrom sklearn.preprocessing import MinMaxScaler, StandardScaler\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import confusion_matrix, accuracy_score, classification_report, cohen_kappa_score\nfrom operator import truediv\nimport copy\n\ndef random_unison(a,b, rstate=None):\n    assert len(a) == len(b)\n    p = np.random.RandomState(seed=rstate).permutation(len(a))\n    return a[p], b[p]\n\n\ndef loadDataFIX(args):\n    data, labels, numclass = loadData(args.dataset, num_components=args.components)\n    num_bands = data.shape[-1]\n    pixels, labels = createImageCubes(data, labels, windowSize=args.spatialsize, removeZeroLabels = False)\n    num_class = len(np.unique(labels)) - 1\n    if args.dataset == "UH":\n        data_path = os.path.join(os.getcwd(),\'HSI-datasets/Classification/\')\n        y_train = sio.loadmat(os.path.join(data_path, \'houston_gt.mat\'))[\'houston_gt_tr\'].reshape(-1)\n        y_test = sio.loadmat(os.path.join(data_path, \'houston_gt.mat\'))[\'houston_gt_te\'].reshape(-1)\n        Xtrain = pixels[y_train!=0,:,:,:]\n        Xtest  = pixels[y_test!=0,:,:,:]\n        del pixels\n        y_train = y_train[y_train!=0] - 1\n        y_test  = y_test[y_test!=0] - 1\n        return Xtrain, Xtest, y_train, y_test, num_bands, num_class\n    elif args.dataset == "DIP":\n        y_train2 = sio.loadmat(os.path.join(\'HSI-datasets/Classification/\', \'indianpines_disjoint_dset.mat\'))\\\n                                            [\'indianpines_disjoint_dset\']\n        y_test = sio.loadmat(os.path.join(\'HSI-datasets/Classification/\', \'indian_pines_corrected_gt.mat\'))[\'indian_pines_gt\']\n        y_train = copy.deepcopy(y_train2)\n        for i, val in enumerate([0,2,3,5,6,8,10,11,12,14,1,4,7,9,13,15,16]): y_train[y_train2==i] = val\n        del y_train2\n        y_test[y_train!=0] = 0\n    elif args.dataset == "DUP":\n        y_train = sio.loadmat(os.path.join(\'HSI-datasets/Classification/\', \'TRpavia_fixed.mat\'))[\'TRpavia_fixed\'].reshape(-1)\n        y_test = sio.loadmat(os.path.join(\'HSI-datasets/Classification/\', \'TSpavia_fixed.mat\'))[\'TSpavia_fixed\'].reshape(-1)\n    y_train = y_train.reshape(-1)\n    y_test = y_test.reshape(-1)\n    Xtrain = pixels[y_train!=0,:]\n    Xtest  = pixels[y_test!=0,:]\n    #del pixels\n    y_train = y_train[y_train!=0] - 1\n    y_test  = y_test[y_test!=0] - 1\n    Xtrain, y_train = random_unison(Xtrain,y_train, rstate=None)\n    return Xtrain, Xtest, y_train, y_test, pixels, labels, num_bands, num_class\n\n\ndef split_data(pixels, labels, percent, splitdset="sklearn", with_val=False, rand_state=None):\n    return train_test_split(pixels, labels, test_size=(1-percent), stratify=labels, random_state=rand_state)\n\ndef loadData(name, num_components=None, rand_state=None):\n    data_path = os.path.join(os.getcwd(),\'HSI-datasets/Classification/\')\n    if name in [\'IP\', \'DIP\']:\n        data = sio.loadmat(os.path.join(data_path, \'indian_pines_corrected.mat\'))[\'indian_pines_corrected\']\n        labels = sio.loadmat(os.path.join(data_path, \'indian_pines_corrected_gt.mat\'))[\'indian_pines_gt\']\n    elif name == \'SV\':\n        data = sio.loadmat(os.path.join(data_path, \'salinas_corrected.mat\'))[\'salinas_corrected\']\n        labels = sio.loadmat(os.path.join(data_path, \'salinas_gt.mat\'))[\'salinas_gt\']\n    elif name in [\'UP\', \'DUP\']:\n        data = sio.loadmat(os.path.join(data_path, \'paviaU.mat\'))[\'paviaU\']\n        labels = sio.loadmat(os.path.join(data_path, \'paviaU_gt.mat\'))[\'paviaU_gt\']\n    elif name == \'KSC\':\n        data = sio.loadmat(os.path.join(data_path, \'KSC.mat\'))[\'KSC\']\n        labels = sio.loadmat(os.path.join(data_path, \'KSC_gt.mat\'))[\'KSC_gt\']\n    elif name == \'UH\':\n        data = sio.loadmat(os.path.join(data_path, \'houston.mat\'))[\'houston\']\n        labels = sio.loadmat(os.path.join(data_path, \'houston_gt.mat\'))[\'houston_gt_tr\']\n        labels += sio.loadmat(os.path.join(data_path, \'houston_gt.mat\'))[\'houston_gt_te\']\n    elif name == \'BW\':\n        data = sio.loadmat(os.path.join(data_path, \'Botswana.mat\'))[\'Botswana\']\n        labels = sio.loadmat(os.path.join(data_path, \'Botswana_gt.mat\'))[\'Botswana_gt\']\n    else:\n        print("NO DATASET")\n        exit()\n\n    shapeor = data.shape\n    data = data.reshape(-1, data.shape[-1])\n    if num_components != None:\n        data = PCA(n_components=num_components, random_state=rand_state).fit_transform(data)\n        shapeor = np.array(shapeor)\n        shapeor[-1] = num_components\n    data = MinMaxScaler(feature_range=(-1, 1)).fit_transform(data)\n    data = data.reshape(shapeor)\n    num_class = len(np.unique(labels)) - 1\n    return data, labels, num_class\n\n\ndef padWithZeros(X, margin=2):\n    newX = np.zeros((X.shape[0] + 2 * margin, X.shape[1] + 2* margin, X.shape[2]))\n    x_offset = margin\n    y_offset = margin\n    newX[x_offset:X.shape[0] + x_offset, y_offset:X.shape[1] + y_offset, :] = X\n    return newX\n\ndef createImageCubes(X, y, windowSize=5, removeZeroLabels = True):\n    margin = int((windowSize - 1) / 2)\n    zeroPaddedX = padWithZeros(X, margin=margin)\n    # split patches\n    patchesData = np.zeros((X.shape[0] * X.shape[1], windowSize, windowSize, X.shape[2])).astype("float32")\n    patchesLabels = np.zeros((X.shape[0] * X.shape[1]))\n    patchIndex = 0\n    for r in range(margin, zeroPaddedX.shape[0] - margin):\n        for c in range(margin, zeroPaddedX.shape[1] - margin):\n            patch = zeroPaddedX[r - margin:r + margin + 1, c - margin:c + margin + 1]\n            patchesData[patchIndex, :, :, :] = patch.astype("float32")\n            patchesLabels[patchIndex] = y[r-margin, c-margin]\n            patchIndex = patchIndex + 1\n    if removeZeroLabels:\n        patchesData = patchesData[patchesLabels>0,:,:,:]\n        patchesLabels = patchesLabels[patchesLabels>0]\n        patchesLabels -= 1\n    return patchesData, patchesLabels.astype("int")\n\n\ndef accuracy(output, target, topk=(1,)):\n    output = np.argmax(output, axis=1)\n    return np.sum(output == target) * 100\n\n\ndef AA_andEachClassAccuracy(confusion_matrix):\n    counter = confusion_matrix.shape[0]\n    list_diag = np.diag(confusion_matrix)\n    list_raw_sum = np.sum(confusion_matrix, axis=1)\n    each_acc = np.nan_to_num(truediv(list_diag, list_raw_sum))\n    average_acc = np.mean(each_acc)\n    return each_acc, average_acc\n\n\ndef reports(y_pred, y_test):\n    classification = classification_report(y_test, y_pred)\n    oa = accuracy_score(y_test, y_pred)\n    confusion = confusion_matrix(y_test, y_pred)\n    each_acc, aa = AA_andEachClassAccuracy(confusion)\n    kappa = cohen_kappa_score(y_test, y_pred)\n\n    return classification, confusion, list(np.round(np.array([oa, aa, kappa] + list(each_acc)) * 100, 2))\n', 'dense_suppli.py': "# For running in python 2.x\nfrom __future__ import print_function, unicode_literals\nfrom __future__ import absolute_import, division\n\nimport sys\nimport numpy as np\nfrom sklearn.metrics import confusion_matrix\nfrom scipy.spatial.distance import cdist\nfrom keras.utils.np_utils import to_categorical\n\ndef relabel(labelTr, labelTs):\n    unqLab, pInClass=np.unique(labelTr, return_counts=True)\n    sortedUnqLab=np.argsort(pInClass, kind='mergesort')\n    c=sortedUnqLab.shape[0]\n    labelsNewTr=np.zeros((labelTr.shape[0],))-1\n    labelsNewTs=np.zeros((labelTs.shape[0],))-1\n    pInClass=np.sort(pInClass)\n    classMap=list()\n    for i in range(c):\n        labelsNewTr[labelTr==unqLab[sortedUnqLab[i]]]=i\n        labelsNewTs[labelTs==unqLab[sortedUnqLab[i]]]=i\n        classMap.append(np.where(labelsNewTr==i)[0])\n    return labelsNewTr, labelsNewTs, c, pInClass, classMap, sortedUnqLab\n\ndef irFind(pInClass, c, irIgnore=1):\n    ir=pInClass[-1]/pInClass\n    imbalancedCls=np.arange(c)[ir>irIgnore]\n    toBalance=np.subtract(pInClass[-1], pInClass[imbalancedCls])\n    imbClsNum=toBalance.shape[0]\n    if imbClsNum==0: sys.exit('No imbalanced classes found, exiting ...')\n    return imbalancedCls, toBalance, imbClsNum, ir\n\ndef fileRead(fileName):\n    dataTotal=np.loadtxt(fileName, delimiter=',')\n    data=dataTotal[:, :-1]\n    labels=dataTotal[:, -1]\n    return data, labels\n\ndef indices(pLabel, tLabel):\n    confMat=confusion_matrix(tLabel, pLabel)\n    nc=np.sum(confMat, axis=1)\n    tp=np.diagonal(confMat)\n    tpr=tp/nc\n    acsa=np.mean(tpr)\n    gm=np.prod(tpr)**(1/confMat.shape[0])\n    acc=np.sum(tp)/np.sum(nc)\n    return acsa, gm, tpr, confMat, acc\n\ndef randomLabelGen(toBalance, batchSize, c):\n    cumProb=np.cumsum(toBalance/np.sum(toBalance))\n    bins=np.insert(cumProb, 0, 0)\n    randomValue=np.random.rand(batchSize,)\n    randLabel=np.digitize(randomValue, bins)-1\n    randLabel_cat=to_categorical(randLabel)\n    labelPadding=np.zeros((batchSize, c-randLabel_cat.shape[1]))\n    randLabel_cat=np.hstack((randLabel_cat, labelPadding))\n    return randLabel_cat\n\ndef batchDivision(n, batchSize):\n    numBatches, residual=int(np.ceil(n/batchSize)), int(n%batchSize)\n    if residual==0:\n        residual=batchSize\n    batchDiv=np.zeros((numBatches+1,1), dtype='int64')\n    batchSizeStore=np.ones((numBatches, 1), dtype='int64')\n    batchSizeStore[0:-1, 0]=batchSize\n    batchSizeStore[-1, 0]=residual\n    for i in range(numBatches):\n        batchDiv[i]=i*batchSize\n    batchDiv[numBatches]=batchDiv[numBatches-1]+residual\n    return batchDiv, numBatches, batchSizeStore\n\ndef rearrange(labelsCat, numImbCls):\n    labels=np.argmax(labelsCat, axis=1)\n    arrangeMap=list()\n    for i in range(numImbCls):\n        arrangeMap.append(np.where(labels==i)[0])\n    return arrangeMap\n", 'README.md': '# Generative Adversarial Minority Oversampling for Spectral-Spatial Hyperspectral Image Classification\nThe Code for "Generative Adversarial Minority Oversampling for Spectral-Spatial Hyperspectral Image Classification". [https://ieeexplore.ieee.org/document/9347550]\n```\nS. K. Roy, J. M. Haut, M. E. Paoletti, S. R. Dubey and A. Plaza. \nGenerative Adversarial Minority Oversampling for Spectral-Spatial Hyperspectral Image Classification\nIEEE Transactions on Geoscience and Remote Sensing\nDOI: 10.1109/TGRS.2021.3052048\nFebruary 2021.\n```\n\n![3DGAMO](./images/3DGAMO.png)\n\n\n### Install and activate requires packages (with conda)\n```\nconda env create -f enviroment.yml\nconda activate 3D-HyperGAMO\n```\n\n### Example of use\n```\n# Without datasets\ngit clone https://github.com/mhaut/3D-HyperGAMO/\n\n# With datasets\ngit clone --recursive https://github.com/mhaut/3D-HyperGAMO/\ncd HSI-datasets\npython join_dsets.py\n```\n\n\n### Run code\n\n```\npython main.py --dataset IP \n\n```\n\nReference code: https://github.com/SankhaSubhra/GAMO\n'}


In [ ]:
WORKER_SOURCE = '"""3D-HyperGAMO training adapter: original networks, MSE and adversarial update sequence.\n\nOnly data protocol, compatibility, logging, validation selection and persistence differ.\nNever import the test cache in the training stage.\n"""\nimport argparse\nimport csv\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport traceback\nimport zipfile\n\nparser = argparse.ArgumentParser()\nparser.add_argument(\'stage\', choices=[\'smoke\', \'train\', \'samples\', \'evaluate\', \'infer\'])\nparser.add_argument(\'--run\', required=True)\nparser.add_argument(\'--input\', help=\'Prepared inference patches; no fitting or labels.\')\nparser.add_argument(\'--output\', help=\'Destination for inference probabilities.\')\nparser.add_argument(\'--cpu\', action=\'store_true\', help=\'CPU only for engineering smoke tests.\')\nargs = parser.parse_args()\nRUN = Path(args.run).resolve()\nCFG = json.loads((RUN / \'config.json\').read_text(encoding=\'utf-8\'))\nos.environ[\'CUDA_VISIBLE_DEVICES\'] = \'-1\' if args.cpu else str(CFG[\'gpu_index\'])\nos.environ[\'TF_CPP_MIN_LOG_LEVEL\'] = \'2\'\nos.environ[\'TF_USE_LEGACY_KERAS\'] = \'1\'\nos.environ[\'PYTHONUNBUFFERED\'] = \'1\'\n\nimport numpy as np\nimport tensorflow as tf\nimport tf_keras as keras\nfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_score, cohen_kappa_score, precision_recall_fscore_support\n\ntf.config.threading.set_intra_op_parallelism_threads(2)\ntf.config.threading.set_inter_op_parallelism_threads(2)\ndevices = tf.config.list_physical_devices(\'GPU\')\nfor dev in devices:\n    tf.config.experimental.set_memory_growth(dev, True)\nif args.stage != \'smoke\' and (not devices or args.cpu):\n    raise RuntimeError(\'Huấn luyện/đánh giá cần GPU. Chưa khởi chạy CPU thay thế.\')\nprint(\'TensorFlow:\', tf.__version__, \'| tf-keras:\', keras.__version__, \'| GPU:\', devices, flush=True)\nkeras.utils.set_random_seed(int(CFG[\'train_seed\']))\ntry:\n    tf.config.experimental.enable_op_determinism()\nexcept AttributeError:\n    pass\n\n# The pinned author source is embedded next to this adapter. Import-only compatibility changes.\nauthor_source = (RUN / \'code\' / \'dense_net3D_SA_original.py\').read_text(encoding=\'utf-8\')\nsource = author_source.replace(\'from keras.layers.advanced_activations import\', \'from tf_keras.layers import\')\nsource = source.replace(\'from keras\', \'from tf_keras\')\nNT = {}\nexec(compile(source, \'dense_net3D_SA_compat.py\', \'exec\'), NT)\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef bundle():\n    target = RUN.parent / (RUN.name + \'.zip\')\n    tmp = target.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(tmp, \'w\', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:\n        for path in sorted(RUN.rglob(\'*\')):\n            if path.is_file() and not path.name.endswith(\'.tmp\'):\n                archive.write(path, str(Path(RUN.name) / path.relative_to(RUN)))\n    tmp.replace(target)\n    print(\'ZIP:\', target, flush=True)\n\n\ndef make_models(x, y, seed):\n    keras.utils.set_random_seed(seed)\n    np.random.seed(seed)\n    ids, counts = np.unique(y, return_counts=True)\n    assert np.array_equal(ids, np.arange(len(ids)))\n    order = np.argsort(counts, kind=\'mergesort\')\n    inverse = np.empty_like(order)\n    inverse[order] = np.arange(len(order))\n    y = inverse[y].astype(np.int64)\n    counts = counts[order]\n    permutation = np.random.choice(np.arange(len(x)), size=len(x), replace=False)\n    x, y = x[permutation], y[permutation]\n    c = len(order)\n    minority = np.flatnonzero(counts < counts[-1])\n    if len(minority) == 0:\n        raise RuntimeError(\'Train đã cân bằng; không có lớp thiểu số cho HyperGAMO.\')\n    assert np.array_equal(minority, np.arange(len(minority)))\n    supports = [x[y == i] for i in minority]\n    shape = tuple(x.shape[1:])\n    # Original code uses ONE shared Adam across C, D and composite G models.\n    opt = keras.optimizers.legacy.Adam(learning_rate=0.0002, beta_1=0.5)\n    C = NT[\'denseMlpCreate\'](sh=shape, num_class=c)\n    C.compile(loss=\'mean_squared_error\', optimizer=opt)\n    C.trainable = False\n    D = NT[\'denseDisCreate\'](shape, num_class=c)\n    D.compile(loss=\'mean_squared_error\', optimizer=opt)\n    D.trainable = False\n    G = NT[\'denseGamoGenCreate\'](100, c)\n    heads, vs_c, vs_d = [], [], []\n    for support in supports:\n        head = NT[\'denseGenProcessCreate\'](len(support), support, sh=shape, mul=int(np.prod(shape)))\n        heads.append(head)\n        z = keras.layers.Input(shape=(100,))\n        label = keras.layers.Input(shape=(c,))\n        mixed = head(G([z, label]))\n        g_c = keras.Model([z, label], C(mixed))\n        g_c.compile(loss=\'mean_squared_error\', optimizer=opt)\n        vs_c.append(g_c)\n        z2 = keras.layers.Input(shape=(100,))\n        label2 = keras.layers.Input(shape=(c,))\n        label3 = keras.layers.Input(shape=(c,))\n        mixed2 = head(G([z2, label2]))\n        g_d = keras.Model([z2, label2, label3], D([mixed2, label3]))\n        g_d.compile(loss=\'mean_squared_error\', optimizer=opt)\n        vs_d.append(g_d)\n    tracked = tf.Module()\n    tracked.classifier, tracked.discriminator, tracked.generator = C, D, G\n    tracked.optimizer = opt\n    for i, head in enumerate(heads):\n        setattr(tracked, f\'head_{i:03d}\', head)\n    checkpoint = tf.train.Checkpoint(models=tracked)\n    return dict(C=C, D=D, G=G, heads=heads, vs_c=vs_c, vs_d=vs_d, checkpoint=checkpoint,\n                optimizer=opt, x=x, y=y, order=order, inverse=inverse, counts=counts,\n                supports=supports, shape=shape, c=c)\n\n\ndef scalar(value):\n    value = float(np.asarray(value).reshape(-1)[0])\n    if not np.isfinite(value):\n        raise FloatingPointError(\'Loss NaN/Inf. Dừng, không ghi kết quả thành công.\')\n    return value\n\n\ndef predict(C, x, batch=32):\n    return np.concatenate([C.predict_on_batch(x[i:i+batch]) for i in range(0, len(x), batch)])\n\n\ndef metrics(y, pred, c):\n    precision, recall, f1, support = precision_recall_fscore_support(\n        y, pred, labels=np.arange(c), zero_division=0)\n    result = dict(OA=float(accuracy_score(y, pred)), AA=float(recall.mean()),\n                  Macro_F1=float(f1.mean()), GM=float(np.exp(np.log(recall).mean())) if (recall > 0).all() else 0.,\n                  Kappa=float(cohen_kappa_score(y, pred)))\n    return result, (precision, recall, f1, support)\n\n\ndef snap(model):\n    return [w.numpy().copy() for w in model.weights]\n\n\ndef changed(model, before):\n    return any(not np.array_equal(w.numpy(), old) for w, old in zip(model.weights, before))\n\n\ndef smoke():\n    # Same spatial/spectral dimensions as the intended run; three artificial classes only.\n    shape = (CFG[\'patch_size\'], CFG[\'patch_size\'], CFG[\'pca_components\'], 1)\n    rng = np.random.default_rng(7)\n    x = rng.normal(size=(9, *shape)).astype(\'float32\')\n    y = np.repeat(np.arange(3), [2, 3, 4])\n    m = make_models(x, y, 7)\n    real = m[\'supports\'][0][:2]\n    labels = keras.utils.to_categorical([0, 0], 3).astype(\'float32\')\n    noise = rng.normal(size=(2, 100)).astype(\'float32\')\n    for name, model, inputs, targets in [\n        (\'C real\', m[\'C\'], real, labels),\n        (\'D real\', m[\'D\'], [real, labels], np.ones((2, 1), \'float32\')),\n    ]:\n        before = snap(model)\n        loss = scalar(model.train_on_batch(inputs, targets))\n        assert changed(model, before), name + \': trọng số không cập nhật.\'\n        print(\'PASS\', name, \'loss\', loss, flush=True)\n    fake = m[\'heads\'][0].predict_on_batch(m[\'G\'].predict_on_batch([noise, labels]))\n    assert fake.shape == real.shape and np.isfinite(fake).all()\n    assert (fake >= real.min(0) - 1e-5).all() and (fake <= real.max(0) + 1e-5).all()\n    for name, model, inputs, targets in [\n        (\'C fake\', m[\'C\'], fake, labels),\n        (\'D fake\', m[\'D\'], [fake, labels], np.zeros((2, 1), \'float32\')),\n    ]:\n        before = snap(model)\n        loss = scalar(model.train_on_batch(inputs, targets))\n        assert changed(model, before), name + \': trọng số không cập nhật.\'\n        print(\'PASS\', name, \'loss\', loss, flush=True)\n    for name, model, inputs, targets in [\n        (\'G against C\', m[\'vs_c\'][0], [noise, labels], 1-labels),\n        (\'G against D\', m[\'vs_d\'][0], [noise, labels, labels], np.ones((2, 1), \'float32\')),\n    ]:\n        cb, db, gb, hb = map(snap, [m[\'C\'], m[\'D\'], m[\'G\'], m[\'heads\'][0]])\n        loss = scalar(model.train_on_batch(inputs, targets))\n        assert not changed(m[\'C\'], cb) and not changed(m[\'D\'], db), name + \': C/D không đóng băng.\'\n        assert changed(m[\'G\'], gb) and changed(m[\'heads\'][0], hb), name + \': G/head không cập nhật.\'\n        print(\'PASS\', name, \'loss\', loss, flush=True)\n    # Checkpoint reload must preserve model output before any long run.\n    smoke_dir = RUN / \'smoke_checkpoint\'\n    smoke_dir.mkdir(exist_ok=True)\n    path = m[\'checkpoint\'].write(str(smoke_dir / \'check\'))\n    expected = m[\'C\'].predict_on_batch(real)\n    first = m[\'C\'].weights[0]\n    first.assign(tf.zeros_like(first))\n    m[\'checkpoint\'].restore(path).expect_partial()\n    np.testing.assert_allclose(m[\'C\'].predict_on_batch(real), expected, rtol=1e-5, atol=1e-6)\n    atomic_json(RUN / \'smoke_pass.json\', dict(passed=True, tensorflow=tf.__version__, tf_keras=keras.__version__,\n                shape=list(shape), gpu=bool(devices), data_is_artificial=True))\n    print(\'SMOKE PASSED — đây không phải kết quả nghiên cứu.\', flush=True)\n\n\ndef write_history(history):\n    path = RUN / \'training_history.csv\'\n    tmp = path.with_suffix(\'.csv.tmp\')\n    with open(tmp, \'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(history[0]))\n        writer.writeheader()\n        writer.writerows(history)\n    tmp.replace(path)\n\n\ndef train():\n    started = time.monotonic()\n    if not (RUN / \'smoke_pass.json\').exists():\n        raise RuntimeError(\'Chưa có smoke test thành công.\')\n    state_path = RUN / \'training_state.json\'\n    old_state = json.loads(state_path.read_text()) if state_path.exists() else None\n    if old_state and old_state[\'complete\']:\n        print(\'Lần chạy này đã hoàn thành. Không train lại.\', flush=True)\n        return\n    data = np.load(RUN / \'train_val.npz\', allow_pickle=False)\n    x, y = data[\'X_train\'], data[\'y_train\']\n    val_x, val_y_orig = data[\'X_val\'], data[\'y_val\']\n    m = make_models(x, y, int(CFG[\'train_seed\']))\n    val_y = m[\'inverse\'][val_y_orig]\n    class_ids = np.asarray(CFG[\'accepted_class_ids\'], dtype=int)[m[\'order\']]\n    atomic_json(RUN / \'model_class_order.json\', dict(original_class_ids=class_ids.tolist()))\n    latest = tf.train.CheckpointManager(m[\'checkpoint\'], str(RUN / \'checkpoints\' / \'latest\'), max_to_keep=2)\n    best = tf.train.CheckpointManager(m[\'checkpoint\'], str(RUN / \'checkpoints\' / \'best\'), max_to_keep=2)\n    use_val = CFG[\'protocol\'] == \'dip_train_only_validation\'\n    if use_val != bool(len(val_x)):\n        raise RuntimeError(\'Cache validation không khớp protocol.\')\n    step, history, best_f1 = 0, [], -1.\n    generated_counts = np.zeros(m[\'c\'], dtype=np.int64)\n    # Roll back metadata too if a prior process died between saving best and latest.\n    if old_state and old_state.get(\'selection\'):\n        atomic_json(RUN / \'selection.json\', old_state[\'selection\'])\n    if (RUN / \'selection.json\').exists():\n        best_f1 = json.loads((RUN / \'selection.json\').read_text()).get(\'Macro_F1\', -1.)\n    if old_state:\n        if not CFG[\'resume_training\']:\n            raise RuntimeError(\'Có checkpoint dở dang. Bật resume_training để tiếp tục, không ghi đè.\')\n        path = RUN / old_state[\'checkpoint\']\n        m[\'checkpoint\'].restore(str(path)).expect_partial()\n        step = int(old_state[\'step\'])\n        generated_counts = np.array(old_state[\'generated_counts\'], dtype=np.int64)\n        state = old_state[\'numpy_rng\']\n        np.random.set_state((state[0], np.asarray(state[1], dtype=np.uint32), state[2], state[3], state[4]))\n        if (RUN / \'training_history.csv\').exists():\n            with open(RUN / \'training_history.csv\', newline=\'\') as f:\n                history = [dict(row) for row in csv.DictReader(f) if int(row[\'step\']) <= step]\n        print(\'Tiếp tục từ step\', step, \'(không cam kết trùng bit với chạy liên tục: RNG Dropout).\', flush=True)\n    labels = keras.utils.to_categorical(m[\'y\'], m[\'c\']).astype(\'float32\')\n    n, bs = len(m[\'x\']), int(CFG[\'batch_size\'])\n    n_batches = int(math.ceil(n / bs))\n    gen_points = int(math.ceil(bs / m[\'c\']))\n    deficits = m[\'counts\'][-1] - m[\'counts\'][:len(m[\'heads\'])]\n    cumulative = np.cumsum(deficits / deficits.sum())\n    max_step = int(CFG[\'max_step\'])\n    if max_step <= 0 or max_step % 2:\n        raise ValueError(\'max_step phải là số chẵn dương (mỗi vòng có hai cập nhật C).\')\n    recent = np.zeros(4, dtype=float)\n    recent_n = 0\n    while step < max_step:\n        j = (step // 2) % n_batches\n        a, b = j * bs, min((j + 1) * bs, n)\n        size = b - a\n        valid_r = 1. - np.random.uniform(0., .1, (size, 1))\n        loss_c_real = scalar(m[\'C\'].train_on_batch(m[\'x\'][a:b], labels[a:b]))\n        loss_d_real = scalar(m[\'D\'].train_on_batch([m[\'x\'][a:b], labels[a:b]], valid_r))\n        invalid = np.random.uniform(0., .1, (size, 1))\n        z = np.random.normal(size=(size, 100))\n        fake_ids = np.searchsorted(cumulative, np.random.rand(size), side=\'right\')\n        generated_counts += np.bincount(fake_ids, minlength=m[\'c\'])\n        fake_labels = keras.utils.to_categorical(fake_ids, m[\'c\']).astype(\'float32\')\n        features = m[\'G\'].predict_on_batch([z, fake_labels])\n        fake = np.zeros((size, *m[\'shape\']), dtype=\'float32\')\n        for i, head in enumerate(m[\'heads\']):\n            idx = np.flatnonzero(fake_ids == i)\n            if len(idx):\n                fake[idx] = head.predict_on_batch(features[idx])\n        loss_c_fake = scalar(m[\'C\'].train_on_batch(fake, fake_labels))\n        loss_d_fake = scalar(m[\'D\'].train_on_batch([fake, fake_labels], invalid))\n        g_c, g_d = [], []\n        for i in range(len(m[\'heads\'])):\n            condition = np.zeros((gen_points, m[\'c\']), dtype=\'float32\')\n            condition[:, i] = 1\n            noise = np.random.normal(size=(gen_points, 100))\n            # Important: retain the original adversarial target, NOT normal class CE.\n            g_c.append(scalar(m[\'vs_c\'][i].train_on_batch([noise, condition], 1-condition)))\n            g_d.append(scalar(m[\'vs_d\'][i].train_on_batch(\n                [noise, condition, condition], np.ones((gen_points, 1)))))\n        recent += [(loss_c_real + loss_c_fake) / 2, (loss_d_real + loss_d_fake) / 2,\n                   np.mean(g_c), np.mean(g_d)]\n        recent_n += 1\n        step += 2\n        timed_out = (time.monotonic() - started) / 60 >= CFG[\'max_minutes\']\n        if step % int(CFG[\'eval_every\']) == 0 or step == max_step or timed_out:\n            values = {}\n            if use_val:\n                val_pred = predict(m[\'C\'], val_x, CFG[\'eval_batch_size\']).argmax(1)\n                values, _ = metrics(val_y, val_pred, m[\'c\'])\n            row = dict(step=step, **values, C_loss=recent[0]/recent_n, D_loss=recent[1]/recent_n,\n                       G_vs_C_loss=recent[2]/recent_n, G_vs_D_loss=recent[3]/recent_n,\n                       elapsed_minutes=(time.monotonic()-started)/60)\n            history.append(row)\n            write_history(history)\n            if (use_val and values[\'Macro_F1\'] > best_f1) or (not use_val and step == max_step):\n                best_f1 = values.get(\'Macro_F1\', -1.)\n                best_path = best.save(checkpoint_number=step)\n                # The metadata must not point to an unfinished/mismatched classifier file.\n                classifier_path = RUN / \'checkpoints\' / \'best\' / f\'classifier_{step}.weights.h5\'\n                temporary = classifier_path.with_name(f\'classifier_{step}.tmp.weights.h5\')\n                m[\'C\'].save_weights(str(temporary))\n                temporary.replace(classifier_path)\n                atomic_json(RUN / \'selection.json\', dict(\n                    step=step, rule=\'validation_macro_f1\' if use_val else \'fixed_final_budget\',\n                    **values, checkpoint=str(Path(best_path).relative_to(RUN)),\n                    classifier_weights=str(classifier_path.relative_to(RUN))))\n            last_path = latest.save(checkpoint_number=step)\n            rng = np.random.get_state()\n            state = dict(step=step, complete=step >= max_step, checkpoint=str(Path(last_path).relative_to(RUN)),\n                         numpy_rng=[rng[0], rng[1].tolist(), int(rng[2]), int(rng[3]), float(rng[4])],\n                         model_class_ids=class_ids.tolist(), generated_counts=generated_counts.tolist(),\n                         selection=json.loads((RUN / \'selection.json\').read_text()) if (RUN / \'selection.json\').exists() else None,\n                         reason=\'time_limit\' if timed_out and step < max_step else \'checkpoint\')\n            atomic_json(state_path, state)\n            atomic_json(RUN / \'online_generated_counts.json\', dict(class_ids=class_ids.tolist(),\n                real_train_counts=m[\'counts\'].tolist(), generated_presentations=generated_counts.tolist(),\n                step=step, note=\'Cumulative fake examples presented to C, not unique saved samples.\'))\n            print(\'VALIDATION\' if use_val else \'TRAIN\', json.dumps({k: v for k, v in row.items() if k not in [\'elapsed_minutes\']}, allow_nan=False), flush=True)\n            bundle()\n            recent[:] = 0\n            recent_n = 0\n        if timed_out and step < max_step:\n            print(\'Hết thời gian đã đặt. Đã lưu checkpoint và ZIP; CHƯA đánh giá test.\', flush=True)\n            return\n    print(\'TRAIN COMPLETE. Chọn checkpoint:\', \'Macro-F1 validation\' if use_val else \'bước cuối cố định\', flush=True)\n\n\ndef evaluate():\n    state = json.loads((RUN / \'training_state.json\').read_text())\n    if not state[\'complete\']:\n        raise RuntimeError(\'Train chưa đủ ngân sách đã đặt; không tự đánh giá test.\')\n    if (RUN / \'test_complete.json\').exists():\n        for name in [\'test_metrics.json\', \'test_per_class.csv\', \'test_predictions.npz\']:\n            if not (RUN / name).is_file():\n                raise RuntimeError(\'Kết quả test thiếu file: \' + name)\n        print(\'Đã có test cuối của run này. Không đánh giá lại.\', flush=True)\n        return\n    # Build only the classifier and load protocol-selected weights.\n    order = json.loads((RUN / \'model_class_order.json\').read_text())[\'original_class_ids\']\n    shape = (CFG[\'patch_size\'], CFG[\'patch_size\'], CFG[\'pca_components\'], 1)\n    C = NT[\'denseMlpCreate\'](shape, len(order))\n    selected = json.loads((RUN / \'selection.json\').read_text())\n    C.load_weights(str(RUN / selected[\'classifier_weights\']))\n    test = np.load(RUN / \'test_locked.npz\', allow_pickle=False)\n    x, original_y = test[\'X_test\'], test[\'original_class_ids\']\n    lookup = {int(cid): i for i, cid in enumerate(order)}\n    y = np.array([lookup[int(cid)] for cid in original_y], dtype=np.int64)\n    prob = predict(C, x, CFG[\'eval_batch_size\'])\n    pred = prob.argmax(1)\n    values, (precision, recall, f1, support) = metrics(y, pred, len(order))\n    atomic_json(RUN / \'test_metrics.json\', dict(**values, selected_step=selected[\'step\'],\n                evaluated_classes=order, protocol=CFG[\'protocol\'], selection_rule=selected[\'rule\'], patch_overlap_expected=True, seed=CFG[\'train_seed\']))\n    with open(RUN / \'test_per_class.csv\', \'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.writer(f)\n        writer.writerow([\'class_id\', \'precision\', \'recall\', \'f1\', \'test_support\'])\n        for i, cid in sorted(enumerate(order), key=lambda item: item[1]):\n            writer.writerow([cid, precision[i], recall[i], f1[i], int(support[i])])\n    np.savez_compressed(RUN / \'test_predictions.npz\', coords=test[\'coords_test\'],\n                        true_class_ids=original_y, predicted_class_ids=np.asarray(order)[pred],\n                        probabilities=prob, probability_class_ids=np.asarray(order),\n                        confusion_matrix=confusion_matrix(y, pred, labels=np.arange(len(order))))\n    atomic_json(RUN / \'test_complete.json\', dict(complete=True, selected_step=selected[\'step\']))\n    print(\'FINAL TEST — DIP, đầy đủ các lớp đã cấu hình; xem audit chồng lấn:\', values, flush=True)\n\n\ndef samples():\n    """Diagnostics use training supports only; no filtering or quality claim is implied."""\n    state = json.loads((RUN / \'training_state.json\').read_text())\n    if not state[\'complete\']:\n        raise RuntimeError(\'Chưa hoàn thành huấn luyện.\')\n    if (RUN / \'samples_complete.json\').exists():\n        for name in [\'generated_examples.npz\', \'generation_diagnostics.csv\']:\n            if not (RUN / name).is_file():\n                raise RuntimeError(\'Minh họa mẫu sinh thiếu file: \' + name)\n        print(\'Đã có mẫu minh họa của checkpoint được chọn.\', flush=True)\n        return\n    cache = np.load(RUN / \'train_val.npz\', allow_pickle=False)\n    m = make_models(cache[\'X_train\'], cache[\'y_train\'], int(CFG[\'train_seed\']))\n    selection = json.loads((RUN / \'selection.json\').read_text())\n    m[\'checkpoint\'].restore(str(RUN / selection[\'checkpoint\'])).expect_partial()\n    ids = np.asarray(CFG[\'accepted_class_ids\'])[m[\'order\']]\n    rng = np.random.default_rng(1701)\n    examples, real_examples, sample_ids, rows, generated_probs = [], [], [], [], []\n    from scipy.spatial.distance import cdist\n    for i, head in enumerate(m[\'heads\']):\n        condition = np.zeros((16, m[\'c\']), dtype=\'float32\')\n        condition[:, i] = 1\n        feature = m[\'G\'].predict_on_batch([rng.normal(size=(16, 100)), condition])\n        fake = head.predict_on_batch(feature)\n        generated_probs.append(predict(m[\'C\'], fake, CFG[\'eval_batch_size\']))\n        weights_model = keras.Model(head.inputs, head.layers[1].output)\n        weights = weights_model.predict_on_batch(feature)\n        h = -(weights * np.log(np.maximum(weights, 1e-12))).sum(axis=1)\n        support = m[\'supports\'][i]\n        distance = cdist(fake.reshape(16, -1), support.reshape(len(support), -1), metric=\'euclidean\')\n        nearest = distance.argmin(axis=1)\n        real_examples.append(support[nearest])\n        examples.append(fake)\n        sample_ids.extend([int(ids[i])] * len(fake))\n        rows.append(dict(class_id=int(ids[i]), real_train=len(support),\n                         normalized_entropy=float((h / np.log(len(support))).mean()),\n                         effective_supports=float(np.exp(h).mean()),\n                         mean_max_weight=float(weights.max(axis=1).mean()),\n                         nearest_real_RMSE=float((distance.min(axis=1) / np.sqrt(np.prod(m[\'shape\']))).mean()),\n                         variance_ratio=float(fake.var(axis=0).mean() / (support.var(axis=0).mean() + 1e-12))))\n    np.savez_compressed(RUN / \'generated_examples.npz\', fake=np.concatenate(examples),\n                        nearest_real=np.concatenate(real_examples), original_class_ids=np.asarray(sample_ids),\n                        probabilities=np.concatenate(generated_probs), probability_class_ids=ids)\n    with open(RUN / \'generation_diagnostics.csv\', \'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    atomic_json(RUN / \'samples_complete.json\', dict(complete=True, selected_step=selection[\'step\']))\n    print(\'Đã lưu mẫu và chẩn đoán theo lớp; chưa phải bằng chứng GAN cải thiện phân loại.\', flush=True)\n\n\n\ndef infer():\n    """Predict user-supplied preprocessed patches with the selected classifier."""\n    if not (RUN / \'test_complete.json\').exists():\n        raise RuntimeError(\'Hoàn thành test cố định trước khi dùng ứng dụng minh họa.\')\n    if not args.input or not args.output:\n        raise ValueError(\'Cần --input và --output cho infer.\')\n    destination = Path(args.output).resolve()\n    if not destination.is_relative_to(RUN):\n        raise ValueError(\'Đầu ra ứng dụng phải nằm trong run.\')\n    x = np.load(args.input, allow_pickle=False)[\'patches\']\n    shape = (CFG[\'patch_size\'], CFG[\'patch_size\'], CFG[\'pca_components\'], 1)\n    if tuple(x.shape[1:]) != shape or len(x) == 0 or not np.isfinite(x).all():\n        raise ValueError(\'Shape hoặc dữ liệu ứng dụng không hợp lệ.\')\n    order = json.loads((RUN / \'model_class_order.json\').read_text())[\'original_class_ids\']\n    C = NT[\'denseMlpCreate\'](shape, len(order))\n    selected = json.loads((RUN / \'selection.json\').read_text())\n    C.load_weights(str(RUN / selected[\'classifier_weights\']))\n    prob = predict(C, x, CFG[\'eval_batch_size\'])\n    np.savez_compressed(destination, probabilities=prob, class_ids=np.asarray(order))\n    print(\'Ứng dụng: đã phân lớp\', len(x), \'vùng ảnh; không cộng vào độ chính xác test.\', flush=True)\n\n\ntry:\n    if args.stage == \'smoke\':\n        smoke()\n    elif args.stage == \'train\':\n        train()\n    elif args.stage == \'samples\':\n        samples()\n    elif args.stage == \'evaluate\':\n        evaluate()\n    else:\n        infer()\nexcept Exception:\n    (RUN / f\'error_{args.stage}.txt\').write_text(traceback.format_exc(), encoding=\'utf-8\')\n    raise\nfinally:\n    # Persist partial results even if a normal Python exception occurs.\n    # A forcibly killed Kaggle VM cannot run this finally block: Save Version is still required.\n    bundle()\n'
compile(WORKER_SOURCE, 'hypergamo_worker.py', 'exec')
print('Đã nạp source; chưa khởi tạo GPU.')


In [ ]:
CNN_WORKER_SOURCE = '"""CNN comparisons for the shared HSI protocol; training never opens test_locked.npz."""\nimport argparse, os, json, sys, time, traceback, hashlib\nfrom pathlib import Path\nparser = argparse.ArgumentParser()\nparser.add_argument(\'stage\', choices=[\'smoke\', \'train\', \'evaluate\'])\nparser.add_argument(\'--run\', required=True)\nparser.add_argument(\'--method\', choices=[\'cnn3d_real_only\', \'cnn3d_random_oversampling\', \'cnn3d_class_weighting\'])\nparser.add_argument(\'--cpu\', action=\'store_true\')\nargs = parser.parse_args()\nRUN = Path(args.run).resolve()\nSETTINGS = json.loads((RUN/\'comparison_config.json\').read_text(encoding=\'utf-8\'))\nCFG = json.loads((RUN/\'config.json\').read_text(encoding=\'utf-8\'))\nos.environ[\'CUDA_VISIBLE_DEVICES\'] = \'-1\' if args.cpu else str(CFG[\'gpu_index\'])\nos.environ[\'TF_USE_LEGACY_KERAS\'] = \'1\'\nos.environ[\'TF_CPP_MIN_LOG_LEVEL\'] = \'2\'\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\nimport tf_keras as keras\nfrom sklearn.metrics import precision_recall_fscore_support, accuracy_score, cohen_kappa_score, confusion_matrix\ntf.config.threading.set_intra_op_parallelism_threads(2)\ntf.config.threading.set_inter_op_parallelism_threads(2)\ndevices = tf.config.list_physical_devices(\'GPU\')\nfor d in devices:\n    tf.config.experimental.set_memory_growth(d, True)\nif not devices and not args.cpu:\n    raise RuntimeError(\'Chưa có GPU. Dừng trước huấn luyện.\')\ntf.config.experimental.enable_op_determinism()\nraw_source = (RUN/\'code/dense_net3D_SA_original.py\').read_text(encoding=\'utf-8\')\nsource = raw_source.replace(\'from keras.layers.advanced_activations import\', \'from tf_keras.layers import\')\nsource = source.replace(\'from keras\', \'from tf_keras\')\nNT = {}\nexec(compile(source, \'author_networks\', \'exec\'), NT)\nBASE = RUN/\'optimized_cnn_baselines\'\nBASE.mkdir(exist_ok=True)\nC = len(CFG[\'accepted_class_ids\'])\nSHAPE = (CFG[\'patch_size\'], CFG[\'pca_components\'])\nSHAPE = (SHAPE[0], SHAPE[0], SHAPE[1], 1)\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\ndef write(path, value):\n    path = Path(path); temporary = path.with_name(path.name+\'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding=\'utf-8\')\n    temporary.replace(path)\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for part in iter(lambda: f.read(4*1024**2), b\'\'): h.update(part)\n    return h.hexdigest()\n\ndef build(lr, seed):\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(seed)\n    model = NT[\'denseMlpCreate\'](SHAPE, C)\n    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),\n                  loss=\'sparse_categorical_crossentropy\', metrics=[\'accuracy\'])\n    return model\n\ndef predict(model, x):\n    bs = int(SETTINGS[\'batch_size\'])\n    return np.concatenate([model.predict_on_batch(x[i:i+bs]) for i in range(0,len(x),bs)])\n\ndef scores(y, prediction):\n    precision, recall, f1, support = precision_recall_fscore_support(\n        y, prediction, labels=np.arange(C), zero_division=0)\n    m = dict(OA=float(accuracy_score(y,prediction)), AA=float(recall.mean()),\n             Macro_F1=float(f1.mean()), GM=float(np.exp(np.log(recall).mean())) if (recall>0).all() else 0.,\n             Kappa=float(cohen_kappa_score(y,prediction)))\n    return m, (precision,recall,f1,support)\n\ndef train_trial(folder, method, seed, lr, cache):\n    folder.mkdir(parents=True, exist_ok=True)\n    state_path = folder/\'state.json\'\n    expected = dict(method=method,seed=int(seed),learning_rate=float(lr),\n                    data_sha256=read(RUN/\'data_ready.json\')[\'train_val_sha256\'], settings=SETTINGS)\n    if (folder/\'identity.json\').exists():\n        if read(folder/\'identity.json\') != expected:\n            raise RuntimeError(\'Trial không cùng cấu hình: \'+str(folder))\n    else: write(folder/\'identity.json\',expected)\n    state = read(state_path) if state_path.exists() else None\n    if state and state[\'complete\']:\n        if not (folder/state[\'best_weights\']).is_file(): raise RuntimeError(\'Thiếu best weights\')\n        return state\n    x,y,vx,vy = (cache[k] for k in [\'X_train\',\'y_train\',\'X_val\',\'y_val\'])\n    counts=np.bincount(y,minlength=C)\n    if (counts==0).any() or not len(vy): raise RuntimeError(\'Thiếu lớp train hoặc validation\')\n    indices=np.arange(len(y)); weights=None\n    if method==\'cnn3d_random_oversampling\':\n        rng=np.random.default_rng(seed)\n        indices=np.concatenate([np.r_[np.flatnonzero(y==c),\n            rng.choice(np.flatnonzero(y==c),int(counts.max()-counts[c]),replace=True)] for c in range(C)])\n        rng.shuffle(indices)\n        np.save(folder/\'ros_indices.npy\',indices)\n    if method==\'cnn3d_class_weighting\':\n        weights={int(c):float(len(y)/(C*counts[c])) for c in range(C)}\n    write(folder/\'data_treatment.json\',dict(real_counts=counts.tolist(),\n          training_counts=np.bincount(y[indices],minlength=C).tolist(),class_weights=weights))\n    model=build(lr,seed)\n    if state:\n        model=keras.models.load_model(str(folder/state[\'latest_model\']))\n        print(\'Tiếp tục\',method,\'seed\',seed,\'sau epoch\',state[\'epoch\'],flush=True)\n    else:\n        state=dict(epoch=0,complete=False,best_score=-1.,best_epoch=0,best_weights=None,\n                   early_wait=0,lr_wait=0,best_val_loss=1e30,history=[],latest_model=None)\n    # Separate streams for deterministic minibatch ordering; Dropout after resume may differ.\n    xf,yf=x[indices],y[indices]\n    for epoch in range(state[\'epoch\']+1,int(SETTINGS[\'max_epochs\'])+1):\n        order=np.random.default_rng(seed*100000+epoch).permutation(len(yf))\n        lr_used=float(keras.backend.get_value(model.optimizer.learning_rate))\n        hist=model.fit(xf[order],yf[order],validation_data=(vx,vy),epochs=1,\n                       batch_size=SETTINGS[\'batch_size\'],class_weight=weights,shuffle=False,verbose=0)\n        metric,_=scores(vy,predict(model,vx).argmax(1))\n        loss=float(hist.history[\'loss\'][-1]); val_loss=float(hist.history[\'val_loss\'][-1])\n        if not np.isfinite([loss,val_loss,*metric.values()]).all(): raise FloatingPointError(\'NaN/Inf\')\n        row=dict(epoch=epoch,loss=loss,val_loss=val_loss,learning_rate=lr_used,\n                 val_macro_f1=metric[\'Macro_F1\'],val_oa=metric[\'OA\'],val_aa=metric[\'AA\'])\n        state[\'history\'].append(row)\n        if metric[\'Macro_F1\']>state[\'best_score\']:\n            old_best=state.get(\'best_weights\')\n            best_name=f\'best_epoch_{epoch}.weights.h5\'\n            model.save_weights(str(folder/best_name))\n            state.update(best_score=metric[\'Macro_F1\'],best_epoch=epoch,best_weights=best_name,early_wait=0)\n            # Chỉ giữ checkpoint tốt nhất hiện tại, tránh làm ZIP phình theo số epoch.\n            if old_best and old_best != best_name:\n                old_path=folder/old_best\n                if old_path.exists(): old_path.unlink()\n        else: state[\'early_wait\']+=1\n        if val_loss < state[\'best_val_loss\']-1e-4:\n            state[\'best_val_loss\']=val_loss; state[\'lr_wait\']=0\n        else: state[\'lr_wait\']+=1\n        if state[\'lr_wait\']>=SETTINGS[\'lr_patience\']:\n            keras.backend.set_value(model.optimizer.learning_rate,max(lr_used*.5,SETTINGS[\'min_lr\']))\n            state[\'lr_wait\']=0\n        state.update(epoch=epoch,complete=(epoch>=SETTINGS[\'max_epochs\'] or\n                      state[\'early_wait\']>=SETTINGS[\'early_patience\']))\n        # Keep current and previous full model. State points only to a fully saved file.\n        old_latest=state[\'latest_model\']; new_latest=f\'latest_epoch_{epoch}.keras\'\n        model.save(str(folder/new_latest))\n        state[\'latest_model\']=new_latest\n        write(state_path,state)\n        pd.DataFrame(state[\'history\']).to_csv(folder/\'history.csv\',index=False)\n        for obsolete in folder.glob(\'latest_epoch_*.keras\'):\n            if obsolete.name not in [old_latest,new_latest]: obsolete.unlink()\n        print(method,\'seed\',seed,\'lr\',lr,\'epoch\',epoch,\'val-F1\',round(metric[\'Macro_F1\'],5),\n              \'best\',state[\'best_epoch\'],flush=True)\n        if state[\'complete\']: break\n    return state\n\ndef train():\n    if not (BASE/\'smoke_pass.json\').exists(): raise RuntimeError(\'Cần chạy kiểm tra CNN trước khi train\')\n    ready=read(RUN/\'data_ready.json\')\n    if digest(RUN/\'train_val.npz\') != ready[\'train_val_sha256\']: raise RuntimeError(\'Cache train đã đổi\')\n    method=args.method\n    if not method: raise ValueError(\'Cần --method\')\n    cache=np.load(RUN/\'train_val.npz\',allow_pickle=False)\n    folder=BASE/method; folder.mkdir(exist_ok=True)\n    first_seed=int(SETTINGS[\'seeds\'][0])\n    trials=[]\n    for i,lr in enumerate(SETTINGS[\'learning_rates\']):\n        trial=folder/\'tuning\'/f\'lr_{i}\'\n        state=train_trial(trial,method,first_seed,float(lr),cache)\n        trials.append(dict(lr=float(lr),score=state[\'best_score\'],folder=str(trial.relative_to(RUN))))\n    winner=max(trials,key=lambda t:t[\'score\'])\n    write(folder/\'tuning.json\',dict(trials=trials,selected=winner,criterion=\'validation_macro_f1\'))\n    for seed in SETTINGS[\'seeds\']:\n        # Reuse the winning trial for its seed, avoiding a duplicate identical train.\n        trial=RUN/winner[\'folder\'] if seed==first_seed else folder/f\'train_seed_{seed}\'\n        state=read(trial/\'state.json\') if seed==first_seed else train_trial(trial,method,int(seed),winner[\'lr\'],cache)\n        output=folder/f\'seed_{seed}\'; output.mkdir(exist_ok=True)\n        write(output/\'selected.json\',dict(seed=int(seed),method=method,learning_rate=winner[\'lr\'],\n              best_epoch=state[\'best_epoch\'],best_val_macro_f1=state[\'best_score\'],complete=state[\'complete\'],\n              classifier_weights=str((trial/state[\'best_weights\']).relative_to(RUN)),\n              history=str((trial/\'history.csv\').relative_to(RUN)),\n              train_val_sha256=ready[\'train_val_sha256\'],test_sha256=ready[\'test_sha256\']))\n    print(\'Đã hoàn thành\',method,flush=True)\n\ndef evaluate():\n    ready=read(RUN/\'data_ready.json\')\n    if digest(RUN/\'test_locked.npz\')!=ready[\'test_sha256\']: raise RuntimeError(\'Cache test đã đổi\')\n    data=np.load(RUN/\'test_locked.npz\',allow_pickle=False)\n    ids=np.array(CFG[\'accepted_class_ids\']); lookup={int(c):i for i,c in enumerate(ids)}\n    y=np.array([lookup[int(c)] for c in data[\'original_class_ids\']])\n    seed_rows=[]; class_rows=[]\n    for method in SETTINGS[\'methods\']:\n        for seed in SETTINGS[\'seeds\']:\n            folder=BASE/method/f\'seed_{seed}\'\n            selection=read(folder/\'selected.json\')\n            if not selection[\'complete\'] or selection[\'test_sha256\']!=ready[\'test_sha256\']:\n                raise RuntimeError(\'Chưa đủ train hoặc sai dữ liệu\')\n            if not (folder/\'test_complete.json\').exists():\n                model=build(selection[\'learning_rate\'],int(seed))\n                model.load_weights(str(RUN/selection[\'classifier_weights\']))\n                prob=predict(model,data[\'X_test\']); prediction=prob.argmax(1)\n                values,(p,r,f,s)=scores(y,prediction)\n                np.savez_compressed(folder/\'test_predictions.npz\',coords=data[\'coords_test\'],\n                    true_class_ids=data[\'original_class_ids\'],predicted_class_ids=ids[prediction],\n                    probabilities=prob,probability_class_ids=ids,seed=np.array(seed),\n                    confusion_matrix=confusion_matrix(y,prediction,labels=np.arange(C)),\n                    test_sha256=np.array(ready[\'test_sha256\']))\n                write(folder/\'test_metrics.json\',dict(**values,seed=int(seed),method=method))\n                pd.DataFrame(dict(Class_ID=ids,Precision=p,Recall=r,F1=f,Test_Support=s)).to_csv(\n                    folder/\'test_per_class.csv\',index=False)\n                write(folder/\'test_complete.json\',dict(complete=True))\n            values=read(folder/\'test_metrics.json\')\n            table=pd.read_csv(folder/\'test_per_class.csv\')\n            seed_rows.append(dict(Method=method,Seed=int(seed),Test_OA=values[\'OA\'],Test_AA=values[\'AA\'],\n                Test_Macro_F1=values[\'Macro_F1\'],Test_Kappa=values[\'Kappa\'],Test_G_mean=values[\'GM\']))\n            table[\'Method\']=method; table[\'Seed\']=seed; class_rows.append(table)\n    pd.DataFrame(seed_rows).to_csv(BASE/\'test_metrics_by_seed.csv\',index=False)\n    pd.concat(class_rows,ignore_index=True).to_csv(BASE/\'test_per_class_all.csv\',index=False)\n    print(pd.DataFrame(seed_rows).to_string(index=False),flush=True)\n\ndef smoke():\n    model=build(2e-4,7)\n    summary=[]; model.summary(print_fn=summary.append)\n    (BASE/\'model_summary.txt\').write_text(\'\\n\'.join(summary),encoding=\'utf-8\')\n    rng=np.random.default_rng(7)\n    x=rng.normal(size=(2,*SHAPE)).astype(\'float32\'); y=np.array([0,1])\n    before=model.weights[0].numpy().copy()\n    loss=model.train_on_batch(x,y)\n    if not np.isfinite(loss).all() or np.array_equal(before,model.weights[0].numpy()):\n        raise RuntimeError(\'CNN không cập nhật đúng\')\n    path=BASE/\'smoke.keras\'; model.save(str(path))\n    expected=predict(model,x); restored=keras.models.load_model(str(path))\n    np.testing.assert_allclose(predict(restored,x),expected,rtol=1e-5,atol=1e-6)\n    if int(model.optimizer.iterations)!=int(restored.optimizer.iterations): raise RuntimeError(\'Optimizer chưa khôi phục\')\n    write(BASE/\'smoke_pass.json\',dict(passed=True,artificial_data=True,gpu=bool(devices)))\n    print(\'CNN SMOKE PASSED\',flush=True)\n\ntry:\n    if args.stage==\'smoke\': smoke()\n    elif args.stage==\'train\': train()\n    else: evaluate()\nexcept BaseException:\n    (BASE/f\'error_{args.stage}_{args.method}.txt\').write_text(traceback.format_exc(),encoding=\'utf-8\')\n    raise\n'
compile(CNN_WORKER_SOURCE, "cnn_worker.py", "exec")
print("Đã nạp mã ba CNN và 3D-HyperGAMO; chưa huấn luyện.")


## 4. Đọc ảnh và nhãn

Mỗi điểm ảnh có 200 giá trị phổ. Nhãn 0 là nền không dùng làm lớp phân loại. Kiểm tra dấu vân tay tệp để không âm thầm dùng nhầm bộ dữ liệu.

In [ ]:
# 4. ĐỌC VÀ KIỂM TRA DỮ LIỆU GỐC
def resolve_input(explicit, filename):
    if explicit:
        p = Path(explicit)
        if not p.is_file(): raise FileNotFoundError(p)
        return p
    if Path('/kaggle/input').is_dir():
        found = sorted(p for p in Path('/kaggle/input').rglob('*.mat') if p.name.lower() == filename.lower())
    else:
        found = [LOCAL_DATA / filename] if (LOCAL_DATA / filename).is_file() else []
    if len(found) != 1:
        raise RuntimeError(f'Tìm được {len(found)} file {filename}: {found}. Điền đường dẫn cụ thể ở ô 1.')
    return found[0]

hsi_path = resolve_input(HSI_PATH, 'Indian_pines_corrected.mat')
gt_path = resolve_input(GT_PATH, 'Indian_pines_gt.mat')
hsi = loadmat(hsi_path)['indian_pines_corrected'].astype(np.float32)
gt = loadmat(gt_path)['indian_pines_gt']
assert hsi.shape == (145, 145, 200) and gt.shape == (145, 145)
assert np.isfinite(hsi).all() and np.issubdtype(gt.dtype, np.integer)
assert np.array_equal(np.unique(gt), np.arange(17)) and int((gt > 0).sum()) == 10249
print('Ảnh:', hsi_path, hsi.shape)
print('Nhãn:', gt_path, '| Có nhãn:', int((gt > 0).sum()))
source_hashes = {'hsi_sha256': digest_file(hsi_path), 'gt_sha256': digest_file(gt_path)}
print('SHA256:', source_hashes)
class_counts = pd.DataFrame({'class_id': np.arange(1, 17), 'class_name': CLASS_NAMES,
                            'total': [int((gt == c).sum()) for c in range(1, 17)]})
display(class_counts)

assert source_hashes['hsi_sha256'] == 'ec2f8808710919d566f70f0d4aa885aae1ddfd42b734aba71c5e12ca65450939', 'Khác file ảnh đã kiểm chứng; cần kiểm tra trước khi gọi là DIP reproduction.'
assert source_hashes['gt_sha256'] == '65c4687a8ab04f6da4789799bc3bc4f6e88bccac3ed6a2e6ae367e5e6b9e429c', 'Khác file nhãn chuẩn đã đối chiếu nguồn tác giả.'


## 5. Nhìn dữ liệu trước khi huấn luyện

Ảnh ba màu chỉ là cách hiển thị ba dải phổ, không phải đầu vào RGB của mô hình. Biểu đồ cho thấy lớp nào ít mẫu; đường phổ biểu diễn thông tin mà mô hình cần học.

In [ ]:
FIGURES = {}
def show_save_later(name, fig):
    FIGURES[name] = fig
    plt.show()

def stretch(a):
    lo, hi = np.percentile(a, [2, 98])
    return np.clip((a - lo) / (hi - lo + 1e-8), 0, 1)

pseudo_rgb = np.stack([stretch(hsi[..., b]) for b in [30, 20, 10]], axis=-1)
class_cmap = ListedColormap([(0, 0, 0, 1)] + [plt.get_cmap('tab20')(i) for i in range(16)])
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].imshow(pseudo_rgb); ax[0].set_title('Ảnh giả màu'); ax[0].axis('off')
ax[1].imshow(gt, cmap=class_cmap, vmin=0, vmax=16); ax[1].set_title('Nhãn chuẩn — 16 lớp'); ax[1].axis('off')
ax[2].bar(class_counts.class_id, class_counts.total)
ax[2].set_xticks(range(1, 17)); ax[2].set_xlabel('ID lớp'); ax[2].set_ylabel('Số pixel có nhãn')
ax[2].set_title('Mất cân bằng trên nhãn chuẩn')
plt.tight_layout(); show_save_later('dataset_overview.png', fig)
print('Nhãn chuẩn: ít nhất/nhiều nhất =', class_counts.total.min(), class_counts.total.max(),
      '| Tỷ lệ =', round(class_counts.total.max() / class_counts.total.min(), 2))
fig, ax = plt.subplots(figsize=(11, 4.5))
for cid in range(1, 17):
    ax.plot(hsi[gt == cid].mean(0), label=f'{cid}: {CLASS_NAMES[cid-1]}')
ax.set_xlabel('Chỉ số dải sau loại bỏ dải nhiễu'); ax.set_ylabel('Giá trị dữ liệu gốc')
ax.set_title('Phổ trung bình từng lớp — chỉ mô tả dữ liệu, không dùng chọn siêu tham số')
ax.legend(ncol=2, fontsize=7); ax.grid(alpha=.2)
plt.tight_layout(); show_save_later('mean_spectra.png', fig)


### 5.1. Khối ảnh siêu phổ

Các lát phổ cho thấy cùng một vùng không gian được quan sát ở nhiều dải; mô hình dùng đồng thời không gian và phổ.

In [ ]:
# Trực quan khối ảnh: hai trục không gian và một trục chỉ số dải phổ.
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
yy, xx = np.mgrid[0:hsi.shape[0]:4, 0:hsi.shape[1]:4]
for band in [0, 30, 60, 90, 120, 150, 199]:
    colors = plt.get_cmap('viridis')(stretch(hsi[..., band])[::4, ::4])
    ax.plot_surface(xx, yy, np.full_like(xx, band), facecolors=colors,
                    rstride=1, cstride=1, shade=False, alpha=.75)
ax.set_xlabel('Cột'); ax.set_ylabel('Hàng'); ax.set_zlabel('Chỉ số dải')
ax.set_title('Một số lát của khối ảnh siêu phổ, không phải địa hình 3D')
ax.view_init(elev=20, azim=-55)
plt.tight_layout(); show_save_later('hsi_cube_3d.png', fig)


## 6. Chia dữ liệu dùng chung cho cả bốn phương pháp

Giữ mặt nạ DIP của nguồn tác giả, lấy 20% theo từng lớp trong phần train làm validation; test giữ cố định. Mặt nạ tác giả có **161 vị trí khác nhãn chuẩn**, được ghi nhận và giữ nguyên để không âm thầm đổi dữ liệu.

**Tâm khác nhau không đồng nghĩa vùng ảnh không chồng lấn.** Ô này in số pixel dùng chung giữa các vùng train/validation/test. PCA chỉ học trên train giúp tránh dùng thống kê test, nhưng không loại bỏ chồng lấn vùng ảnh. Vì vậy kết quả này là so sánh trên cùng giao thức, không chứng minh khả năng tổng quát sang vùng hoàn toàn mới.

In [ ]:
DIP_B64 = 'eJzt201vmzAYwHHDWvKyjeS6G7tvUxupk5JbutOkRZq23KtIoRpTl0RNJvVr7BuPBAjEL/AYG9sk/kuozQvwkwvFNOpsOv82vQ/uPt0Es+n842P0FAbfnxa7x/Xzn0mwWW+jlw/Bl+dwsQuXwXo1Cea//gY/w00w+hzcjiej8eRuFIxubseoRs7XmR9/efmBkBd/7cZLL33pOn38L12ceHkXL9FqGS1Wm2gVbh+W0fb3OlrtHpbbcJeu58bL+3h77r46JqxXSeIbciWJJIPERbJBLNHrNOzpbtzJBuSDGCKNoEzkHMJBmEgfKBedoAwB5apLAhFhIEJ08SAHB+Ei/SAEAiH5INcwUHqeUUCUFIBcs0CF30RGgFw9ID/fAo2TPVsEsSYCUkA+A4RfPEgQKZIB8hkgVxPIZ4BcAoRIECEqgLI4QX4joLf7aoF8HhAiQbiIAuLKZ4O05MNAfAfl1Wn0N3VoiYAcJ7+UXJUmE0ScMHRQuUgeiHIGawF5SYkIBioVoWRXhoGKOxUDecfKQKVR910LhCyIDUqeAB9DqkCHMz/JOFAXsykEFeMAkRvPpis8oNMYswmqiBwhJqh+PNOPFoAov5EtCNuhetBwaBJouK8UhBSChmmmgIYwEDJthHBQgwFBlMsE8wVK2R9IIbddZoIKj1lXe6Wg4o2RdhDaD0/hcs8CUachYBBvrQTRpiFtA3E7+EC0O1W5DFGQ9HHhBQFn9QpB6mLdNWrLgqo6dxDwz5EAkCSRBSkCycHUBg0GBoEGh4wBDbLkCcRqF8irCh1nIYyJAf8RLQbyeEEcI0QTQUCIA1TNEQblH0UAQBBPOQggqwDxNwCJAENkQVUiJaDy47lxECkCgTwZoF6cGaBeHgDEUy1Qr9cukJDIgrSAREQtApVMQSxIGCRDZAyoThSQjPp9paDSN/eTzAEh40BIH8ihlYo0gKic5Bw3DmTUWSaYJBA5RhZ0yaDqz9wUH9RQEHnvfNqbOP7PD2GgDmWltoJ4RQKgqowDofR/XxoBdciVwCA+0TmBKn+yjYI6xEpwEJeoWVCNKCDgShJ2fr6gJqsLuqanDcTwaAOxPPJAfCsxPQ2DWGPH9kgDlWLxd5R4FIIK72kWVI0lx0gv6PidASCbzWaz2Ww2pf0H3l+xCA=='
import base64, zlib, io
dip_bytes = zlib.decompress(base64.b64decode(DIP_B64))
assert hashlib.sha256(dip_bytes).hexdigest() == '430c37ce56a160d410378aa136e5e696e2436e2fe03e275f306c3502f622e77b'
encoded_dip = loadmat(io.BytesIO(dip_bytes))['indianpines_disjoint_dset']
split, labels_protocol, coords_by_split, label_audit = dip_split(gt, encoded_dip, CFG)
overlap_report = audit_overlap(split, CFG['patch_size'])
print('Bất đồng nhãn:', label_audit)
print('Chồng lấn:', json.dumps(overlap_report, indent=2))
count_table = class_counts.copy()
for k, name in enumerate(['train', 'validation', 'test']):
    count_table[name] = [int(((split == k) & (labels_protocol == c)).sum()) for c in range(1, 17)]
assert (count_table.train > 0).all() and (count_table.test > 0).all()
count_table['class_deficit'] = count_table.train.max() - count_table.train
display(count_table)
print('Tổng train/val/test:', [len(c) for c in coords_by_split])
print('Tỷ lệ mất cân bằng TRAIN:', round(count_table.train.max()/count_table.train.min(), 2))
print('CẢNH BÁO: DIP trong notebook này có chồng lấn vùng ảnh; không gọi là leakage-free.')
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(split + 1, cmap=ListedColormap(['black', '#2ca02c', '#ffbf00', '#1f77b4']), vmin=0, vmax=3)
ax[0].set_title('Xanh lá: train; vàng: val; xanh dương: test'); ax[0].axis('off')
ax[1].bar(count_table.class_id, count_table.train, label='Train thật')
ax[1].bar(count_table.class_id, count_table.class_deficit, bottom=count_table.train,
          alpha=.4, label='Thiếu hụt so với lớp lớn nhất')
ax[1].set_xticks(range(1, 17)); ax[1].legend(); ax[1].set_title('Thiếu hụt quyết định xác suất sinh theo lớp')
plt.tight_layout(); show_save_later('dip_split_and_imbalance.png', fig)


## 7. Tạo thư mục nghiên cứu và lưu cấu hình

Thay đổi dữ liệu, cách chia, mã huấn luyện hoặc cấu hình sẽ tạo định danh mới; không ghi đè kết quả nghiên cứu khác.

In [ ]:
# 8. THƯ MỤC RIÊNG CÓ ĐỊNH DANH — KHÔNG TRỘN KẾT QUẢ CŨ
identity = {
    # Runtime controls may change when resuming; model/split/budget must not change.
    'config': {k: v for k, v in CFG.items() if k not in ['max_minutes', 'gpu_index', 'resume_training']},
    'source_data': source_hashes,
    'dip_sha256': hashlib.sha256(dip_bytes).hexdigest(),
    'numeric_versions': {p: importlib.metadata.version(p) for p in ['numpy', 'scipy', 'scikit-learn', 'tensorflow', 'tf-keras']},
    'author_commit': '3f92889aec4a7e2c8f7a2db4c8a9bf167f296f5e',
    'networks_sha256': hashlib.sha256(AUTHOR_NETWORKS.encode()).hexdigest(),
    'worker_sha256': hashlib.sha256(WORKER_SOURCE.encode()).hexdigest(),
    'data_code_sha256': hashlib.sha256(DATA_SOURCE.encode()).hexdigest(),
    'comparison': COMPARISON,
    'cnn_worker_sha256': hashlib.sha256(CNN_WORKER_SOURCE.encode()).hexdigest(),
    'split_sha256': hashlib.sha256(split.tobytes()).hexdigest(),
}
run_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:12]
RUN = OUTPUT_ROOT / f'comparison_4methods_{run_id}'

if RESUME_ZIP:
    # Chỉ ZIP do notebook này tạo. Không thực thi mã lấy từ ZIP; mã hiện tại được ghi lại bên dưới.
    import zipfile
    if RUN.exists():
        raise RuntimeError('Đã có thư mục run. Đặt RESUME_ZIP=None để dùng dữ liệu đang có, tránh ghi đè.')
    with zipfile.ZipFile(RESUME_ZIP) as archive:
        for entry in archive.infolist():
            relative = Path(entry.filename)
            if relative.is_absolute() or '..' in relative.parts or relative.parts[0] != RUN.name:
                raise RuntimeError('ZIP sai cấu hình hoặc có đường dẫn không hợp lệ.')
            target = (OUTPUT_ROOT / relative).resolve()
            if not target.is_relative_to(RUN.resolve()):
                raise RuntimeError('Đường dẫn ngoài run.')
        archive.extractall(OUTPUT_ROOT)

RUN.mkdir(parents=True, exist_ok=True)
manifest_path = RUN / 'manifest.json'
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert saved['identity'] == identity, 'Run khác cấu hình hoặc dữ liệu.'
else:
    manifest_path.write_text(json.dumps({'identity': identity, 'overlap': overlap_report,
        'label_audit': label_audit, 'excluded_classes': [],
        'note': '16-class author DIP; overlapping input patches; not a spatial leakage-free benchmark'},
        ensure_ascii=False, indent=2), encoding='utf-8')
(RUN / 'config.json').write_text(json.dumps(CFG, indent=2), encoding='utf-8')
(RUN / 'code').mkdir(exist_ok=True)
for name, source in [('dip_data.py', DATA_SOURCE), ('dense_net3D_SA_original.py', AUTHOR_NETWORKS),
                     ('main_original.py', AUTHOR_MAIN), ('hypergamo_worker.py', WORKER_SOURCE)]:
    (RUN / 'code' / name).write_text(source, encoding='utf-8')
for name, source in AUTHOR_REFERENCE_FILES.items():
    (RUN / 'code' / ('author_' + name)).write_text(source, encoding='utf-8')
(RUN / 'indianpines_disjoint_dset.mat').write_bytes(dip_bytes)
np.savez_compressed(RUN / 'split_indices.npz', split=split, labels_protocol=labels_protocol,
                    coords_train=coords_by_split[0], coords_val=coords_by_split[1], coords_test=coords_by_split[2],
                    standard_gt=gt, class_ids=np.arange(1, 17))
count_table.to_csv(RUN / 'split_counts.csv', index=False)
for filename, fig in FIGURES.items():
    fig.savefig(RUN / filename, dpi=140, bbox_inches='tight')
    plt.close(fig)
FIGURES.clear()
versions = {p: importlib.metadata.version(p) for p in ['tensorflow', 'tf-keras', 'numpy', 'scipy', 'scikit-learn', 'pandas', 'joblib']}
(RUN / 'environment.json').write_text(json.dumps({'python': platform.python_version(), **versions}, indent=2))
print('Run:', RUN)

(RUN/'comparison_config.json').write_text(json.dumps(COMPARISON,indent=2),encoding='utf-8')
(RUN/'code/cnn_worker.py').write_text(CNN_WORKER_SOURCE,encoding='utf-8')
FIGDIR=RUN/'figures'; FIGDIR.mkdir(exist_ok=True)
def show_fig(name,fig):
    fig.savefig(FIGDIR/(name+'.png'),dpi=160,bbox_inches='tight')
    plt.show(); plt.close(fig)
def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def write_json(path,obj):
    p=Path(path); tmp=p.with_name(p.name+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8'); tmp.replace(p)


## 8. PCA và chuẩn hóa

Học PCA 40 thành phần và MinMax từ **tâm train**, sau đó áp dụng cùng phép biến đổi cho toàn ảnh. Không ép giá trị validation/test về [-1,1] bằng cách học lại trên các tập đó.

In [ ]:
train_coords = coords_by_split[0]
flat = hsi.reshape(-1, hsi.shape[-1]).astype('float64')
fit_spectra = flat if CFG['protocol'] == 'dip_reproduction' else hsi[train_coords[:, 0], train_coords[:, 1]].astype('float64')
if (RUN / 'preprocessing.joblib').exists():
    preprocessing = joblib.load(RUN / 'preprocessing.joblib')
    pca, scaler = preprocessing['pca'], preprocessing['scaler']
else:
    pca = PCA(n_components=CFG['pca_components'], svd_solver=CFG['pca_solver'])
    fit_features = pca.fit_transform(fit_spectra)
    scaler = MinMaxScaler(feature_range=(-1, 1)).fit(fit_features)
    preprocessing = dict(pca=pca, scaler=scaler, raw_bands=hsi.shape[-1],
                         patch_size=CFG['patch_size'], class_ids=CFG['accepted_class_ids'],
                         protocol=CFG['protocol'], fit_count=len(fit_spectra))
    joblib.dump(preprocessing, RUN / 'preprocessing.joblib')
assert pca.n_samples_ == len(fit_spectra)
np.testing.assert_allclose(pca.mean_, fit_spectra.mean(0), rtol=1e-6, atol=1e-5)
processed = scaler.transform(pca.transform(flat)).reshape(145, 145, CFG['pca_components']).astype('float32')
print('Fit PCA/scaler từ:', 'toàn cảnh' if CFG['protocol'] == 'dip_reproduction' else 'tâm train')
print('Số phổ dùng fit:', len(fit_spectra), '| Phương sai giữ lại:', pca.explained_variance_ratio_.sum())
print('Shape:', processed.shape, '| Miền giá trị:', processed.min(), processed.max())
print('Nhánh train-only có thể cho val/test ngoài [-1,1]; không fit lại theo test.')
del flat, fit_spectra


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(np.arange(1,41),np.cumsum(pca.explained_variance_ratio_)*100)
ax[0].set(xlabel='Số thành phần PCA',ylabel='Phương sai giữ lại (%)',title='Thông tin giữ lại sau giảm chiều')
ax[1].imshow(processed[:,:,0],cmap='viridis'); ax[1].set_title('Thành phần PCA thứ nhất'); ax[1].axis('off')
fig.tight_layout(); show_fig('08_pca',fig)


## 9. Cắt vùng ảnh 11 × 11 × 40

Nhãn của mỗi vùng ảnh là nhãn điểm tâm. Ngoài biên ảnh được đệm số 0. Test lưu riêng, chưa dùng để huấn luyện hoặc chọn mô hình.

In [ ]:
# 10. TRÍCH VÙNG ẢNH, LƯU TRAIN/VAL RIÊNG VỚI TEST
class_to_index = {cid: i for i, cid in enumerate(CFG['accepted_class_ids'])}
prepared = RUN / 'data_ready.json'
if not prepared.exists():
    datasets = []
    for coords in coords_by_split:
        x = extract_patches(processed, coords, CFG['patch_size'])
        original_labels = labels_protocol[coords[:, 0], coords[:, 1]].astype('int64')
        y = np.array([class_to_index[int(c)] for c in original_labels], dtype='int64')
        datasets.append((x, y, original_labels))
    np.savez_compressed(RUN / 'train_val.npz', X_train=datasets[0][0], y_train=datasets[0][1],
                        X_val=datasets[1][0], y_val=datasets[1][1])
    np.savez_compressed(RUN / 'test_locked.npz', X_test=datasets[2][0],
                        original_class_ids=datasets[2][2], coords_test=coords_by_split[2])
    ready = {'shapes': [list(d[0].shape) for d in datasets],
             'train_val_sha256': digest_file(RUN / 'train_val.npz'),
             'test_sha256': digest_file(RUN / 'test_locked.npz'),
             'preprocessing_sha256': digest_file(RUN / 'preprocessing.joblib')}
    prepared.write_text(json.dumps(ready, indent=2))
    del datasets
ready = json.loads(prepared.read_text())
for filename, key in [('train_val.npz', 'train_val_sha256'), ('test_locked.npz', 'test_sha256'),
                      ('preprocessing.joblib', 'preprocessing_sha256')]:
    assert digest_file(RUN / filename) == ready[key], f'Cache thay đổi: {filename}'
print('Shape train/validation/test:', ready['shapes'])
del processed


In [ ]:
with np.load(RUN/'train_val.npz') as d:
    small_x=d['X_train'][:6]; small_y=d['y_train'][:6]
fig,axs=plt.subplots(1,6,figsize=(15,3))
for i,a in enumerate(axs):
    a.imshow(small_x[i,:,:,0,0],cmap='viridis'); a.scatter([5],[5],s=20,c='red')
    a.set_title(f'Lớp {small_y[i]+1}'); a.axis('off')
fig.suptitle('Ví dụ vùng train — thành phần PCA thứ nhất, chấm đỏ là tâm')
show_fig('09_train_patches',fig)
del small_x,small_y


## 10. Bốn phương pháp khác nhau ở đâu?

|Phương pháp|Xử lý mất cân bằng|Hàm mất mát / cách học|
|---|---|---|
|CNN thường|Không tăng mẫu, không đổi trọng số|Entropy chéo|
|CNN + ROS|Lặp mẫu train đến số lượng lớp lớn nhất|Entropy chéo|
|CNN + trọng số lớp|Mẫu lớp ít đóng góp trọng số lớn hơn|Entropy chéo có trọng số|
|3D-HyperGAMO|Sinh tổ hợp lồi từ mẫu train cùng lớp; ưu tiên lớp thiếu mẫu|Giữ MSE và đối kháng G–C–D của mã gốc|

Ba CNN dùng **đúng kiến trúc bộ phân lớp của 3D-HyperGAMO**. Cả bốn chọn checkpoint bằng Macro-F1 validation, không chọn bằng test. Ba CNN thử cùng hai tốc độ học, giảm tốc độ học và dừng sớm.

Đây là so sánh **bốn quy trình huấn luyện**, không phải thí nghiệm chỉ thay đúng một yếu tố: GAN và CNN còn khác hàm mất mát, số lần cập nhật, cách tối ưu. Không khẳng định GAN tốt/kém chỉ do cách sinh mẫu từ bảng này.

Kiến trúc bộ phân lớp: đầu vào → 3 lớp tích chập 3D (8, 16, 32 bộ lọc) → chuyển dạng → tích chập 2D (64) → gộp trung bình → Dropout 0,5 → xác suất 16 lớp.

In [ ]:
METHODS=COMPARISON['methods']+['hypergamo']
NAMES={'cnn3d_real_only':'CNN thường','cnn3d_random_oversampling':'CNN + ROS',
       'cnn3d_class_weighting':'CNN + trọng số lớp','hypergamo':'3D-HyperGAMO'}
SEEDS=COMPARISON['seeds']
GAN_RUNS={}
for seed in SEEDS:
    child=RUN/'gan'/f'seed_{seed}'; (child/'code').mkdir(parents=True,exist_ok=True)
    child_cfg={**CFG,'train_seed':seed,'resume_training':True}
    write_json(child/'config.json',child_cfg)
    for filename in ['train_val.npz','test_locked.npz','data_ready.json','split_indices.npz','preprocessing.joblib']:
        dst=child/filename
        if not dst.exists():
            try: os.link(RUN/filename,dst)
            except OSError: shutil.copy2(RUN/filename,dst)
        assert digest_file(dst)==digest_file(RUN/filename), 'GAN dùng cache khác CNN'
    for filename in ['hypergamo_worker.py','dense_net3D_SA_original.py']:
        shutil.copy2(RUN/'code'/filename,child/'code'/filename)
    GAN_RUNS[seed]=child
def zip_run():
    target=OUTPUT_ROOT/(RUN.name+'.zip'); tmp=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for p in sorted(RUN.rglob('*')):
            if p.is_file() and p.suffix not in ['.zip','.tmp']:
                z.write(p,str(Path(RUN.name)/p.relative_to(RUN)))
    tmp.replace(target); return target
def run_worker(kind,stage,method=None,child=None):
    base=child if kind=='gan' else RUN
    script='hypergamo_worker.py' if kind=='gan' else 'cnn_worker.py'
    command=[sys.executable,'-u',str(base/'code'/script),stage,'--run',str(base)]
    if method: command+=['--method',method]
    logpath=base/(stage+('_'+method if method else '')+'.log')
    print('Bắt đầu:',kind,stage,method or base.name,flush=True)
    env={**os.environ,'PYTHONUNBUFFERED':'1','PYTHONIOENCODING':'utf-8'}
    with logpath.open('a',encoding='utf-8') as log:
        p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,encoding='utf-8',errors='replace',env=env)
        try:
            for line in p.stdout: log.write(line); log.flush(); print(line,end='',flush=True)
            status=p.wait()
        except BaseException:
            p.terminate()
            try: p.wait(timeout=20)
            except subprocess.TimeoutExpired: p.kill(); p.wait()
            raise
        finally: print('Đã cập nhật ZIP:',zip_run(),flush=True)
    if status: raise RuntimeError(f'Lỗi {kind}/{stage}; đọc {logpath}. Không tự huấn luyện lại từ đầu.')
def selected_dir(method,seed):
    return GAN_RUNS[seed] if method=='hypergamo' else RUN/'optimized_cnn_baselines'/method/f'seed_{seed}'
print('Đã tạo dữ liệu dùng chung. ZIP:',zip_run())


## 11. Kiểm tra ngắn trước khi huấn luyện

Dùng dữ liệu giả lập kỹ thuật để kiểm tra cập nhật trọng số, luồng đối kháng, lưu và khôi phục mô hình. Kết quả kiểm tra này **không phải kết quả nghiên cứu**.

In [ ]:
if RUN_TRAINING:
    if not (RUN/'optimized_cnn_baselines/smoke_pass.json').exists(): run_worker('cnn','smoke')
    for child in GAN_RUNS.values():
        if not (child/'smoke_pass.json').exists(): run_worker('gan','smoke',child=child)
    print('Đã qua kiểm tra kỹ thuật. Bắt đầu huấn luyện ở các ô tiếp theo.')
else: print('Chỉ chuẩn bị dữ liệu; chưa chạy kiểm tra GPU.')


In [ ]:
def plot_cnn_training(method):
    folder=RUN/'optimized_cnn_baselines'/method
    if not (folder/'tuning.json').exists():
        print('Chưa có kết quả huấn luyện',NAMES[method]); return
    tuning=read_json(folder/'tuning.json')
    display(pd.DataFrame(tuning['trials'])[['lr','score']].rename(columns={'lr':'Tốc độ học','score':'Macro-F1 validation tốt nhất'}))
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    for seed in SEEDS:
        sel=read_json(folder/f'seed_{seed}/selected.json'); h=pd.read_csv(RUN/sel['history'])
        ax[0].plot(h.epoch,h.loss,label=f'Train {seed}'); ax[0].plot(h.epoch,h.val_loss,'--',label=f'Val {seed}')
        ax[1].plot(h.epoch,h.val_macro_f1,label=f'Seed {seed}'); ax[1].axvline(sel['best_epoch'],color='gray',alpha=.4)
    ax[0].set_title('Hàm mất mát'); ax[1].set_title('Macro-F1 validation; vạch đứng = checkpoint chọn')
    for a in ax: a.set_xlabel('Epoch'); a.legend(); a.grid(alpha=.2)
    fig.suptitle(NAMES[method]); fig.tight_layout(); show_fig(method+'_learning',fig)


## 12. CNN thường

Mỗi mẫu train xuất hiện một lần mỗi epoch. Đây là mốc so sánh khi chưa xử lý mất cân bằng. Mô hình tốt nhất được chọn bằng validation, không bằng test.

In [ ]:
if RUN_TRAINING: run_worker('cnn','train',method='cnn3d_real_only')
plot_cnn_training('cnn3d_real_only')


## 13. CNN + lặp mẫu ngẫu nhiên (ROS)

Chỉ lặp mẫu trong train để các lớp có số mẫu bằng nhau. Không tạo thông tin mới và không tăng mẫu validation/test. Mô hình tốt nhất được chọn bằng validation, không bằng test.

In [ ]:
# DÁN TOÀN BỘ FILE VÀO MỘT CELL NGAY TRƯỚC CELL CNN + ROS BỊ LỖI.
# Cell này chỉ chuẩn bị cách tiếp tục; không tự huấn luyện, không mở tập test.
# Sau khi thấy SẴN SÀNG, chạy lại cell CNN + ROS rồi các cell phía sau.
from pathlib import Path
import ast
import hashlib
import json
import math
import os
import subprocess
import sys

_FIX_TAG = "cnn_isolated_trials_batch64_v1"
_FIX_ROOT = Path(globals().get("RUN", "/kaggle/working/comparison_4methods_856313ccab96")).resolve()
if not (_FIX_ROOT / "comparison_config.json").is_file():
    raise RuntimeError("Không thấy kết quả cũ. Khôi phục ZIP và các cell chuẩn bị trước; không Run All để train lại.")
if not callable(globals().get("run_worker")) or not callable(globals().get("zip_run")):
    raise RuntimeError("Thiếu run_worker/zip_run. Cần các cell định nghĩa của notebook cũ trong phiên hiện tại.")


def _fix_read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _fix_write(path, value):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)


def _fix_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(4 * 1024**2), b""):
            h.update(block)
    return h.hexdigest()


_FIX_SETTINGS = _fix_read(_FIX_ROOT / "comparison_config.json")
_FIX_READY = _fix_read(_FIX_ROOT / "data_ready.json")
if _FIX_SETTINGS["batch_size"] != 64:
    raise RuntimeError("Bản khôi phục này dành cho batch 64. Không tự đổi cấu hình đang lưu.")
if not (_FIX_ROOT / "optimized_cnn_baselines/smoke_pass.json").is_file():
    raise RuntimeError("Chưa thấy kiểm tra CNN thành công của lần chạy cũ.")
if _fix_hash(_FIX_ROOT / "train_val.npz") != _FIX_READY["train_val_sha256"]:
    raise RuntimeError("Dữ liệu train/validation không khớp lần chạy cũ.")

# Tạo worker phụ từ CHÍNH source của lần chạy này, không thay source gốc.
_fix_original = _FIX_ROOT / "code/cnn_worker.py"
_fix_source = _fix_original.read_text(encoding="utf-8")
_fix_tree = ast.parse(_fix_source)
_fix_trial = next(n for n in _fix_tree.body if isinstance(n, ast.FunctionDef) and n.name == "train_trial")

# Khi resume: chỉ load model + optimizer, không build thêm một model trước đó.
_fix_builds = [n for n in _fix_trial.body if isinstance(n, ast.Assign)
               and ast.unparse(n) == "model = build(lr, seed)"]
_fix_resume = [n for n in _fix_trial.body if isinstance(n, ast.If)
              and ast.unparse(n.test) == "state"
              and any("keras.models.load_model" in ast.unparse(s) for s in n.body)]
if len(_fix_builds) != 1 or len(_fix_resume) != 1:
    raise RuntimeError("Source khác cấu trúc đã kiểm tra. Dừng an toàn; gửi lại cnn_worker.py, không tự sửa đoán.")
_fix_trial.body.remove(_fix_builds[0])
_fix_resume[0].orelse.insert(0, _fix_builds[0])
_fix_resume[0].body[:0] = ast.parse(
    "keras.backend.clear_session()\nkeras.utils.set_random_seed(seed)"
).body

# Ghi đủ độ đo validation cho các epoch tiếp theo; không dùng test.
_fix_rows = [n for n in ast.walk(_fix_trial) if isinstance(n, ast.Assign)
             and any(isinstance(t, ast.Name) and t.id == "row" for t in n.targets)
             and isinstance(n.value, ast.Call) and ast.unparse(n.value.func) == "dict"]
if len(_fix_rows) != 1:
    raise RuntimeError("Không xác định được dòng ghi lịch sử; dừng trước huấn luyện.")
for _key, _metric in [("val_gm", "GM"), ("val_kappa", "Kappa")]:
    if _key not in [k.arg for k in _fix_rows[0].value.keywords]:
        _fix_rows[0].value.keywords.append(ast.keyword(arg=_key, value=ast.parse(f"metric[{_metric!r}]", mode="eval").body))

# Một số bản cũ xóa best checkpoint trước khi lưu state. Bỏ thao tác đó;
# không xóa bất kỳ checkpoint cũ nào trong cell khôi phục này.
class _FixKeepBest(ast.NodeTransformer):
    def visit_If(self, node):
        if "old_best" in ast.unparse(node.test) and "old_path.unlink()" in ast.unparse(node):
            return None
        return self.generic_visit(node)


_fix_trial = _FixKeepBest().visit(_fix_trial)
if not isinstance(_fix_tree.body[-1], ast.Try) or "args.stage" not in ast.unparse(_fix_tree.body[-1]):
    raise RuntimeError("Không nhận diện được phần chạy chính của worker; chưa thay đổi gì.")
_fix_tree.body.pop()  # bỏ bộ điều phối cũ; vẫn giữ nguyên các hàm thuật toán
_fix_child_tail = r'''
task = json.loads(os.environ['KLTN_CNN_ISOLATED_TASK'])
folder = (RUN / task['folder']).resolve()
folder.relative_to(BASE.resolve())
print('Tiến trình riêng:', os.getpid(), '|', task['action'], '|', task['folder'], flush=True)
if digest(RUN/'train_val.npz') != read(RUN/'data_ready.json')['train_val_sha256']:
    raise RuntimeError('Cache train/validation đã đổi')
with np.load(RUN/'train_val.npz', allow_pickle=False) as cache:
    if task['action'] == 'trial':
        train_trial(folder, task['method'], int(task['seed']), float(task['lr']), cache)
    elif task['action'] == 'validation_history':
        state = read(folder/'state.json')
        model = build(float(task['lr']), int(task['seed']))
        model.load_weights(str(folder/state['best_weights']))
        values, _ = scores(cache['y_val'], predict(model, cache['X_val']).argmax(1))
        row = next(r for r in state['history'] if r['epoch'] == state['best_epoch'])
        for key, metric in [('val_oa','OA'), ('val_aa','AA'), ('val_macro_f1','Macro_F1')]:
            if not np.isclose(float(row[key]), values[metric], atol=1e-6, rtol=0):
                raise RuntimeError('Validation không khớp checkpoint cũ: '+key)
        # Chỉ bổ sung hai số liệu còn thiếu ở BEST epoch, không bịa lịch sử.
        for name in ['state.json', 'history.csv']:
            original = folder/name
            backup = folder/(name+'.before_validation_repair')
            if original.exists() and not backup.exists():
                backup.write_bytes(original.read_bytes())
        row.update(val_gm=values['GM'], val_kappa=values['Kappa'])
        write(folder/'state.json', state)
        history_temp = folder/'history.csv.tmp'
        pd.DataFrame(state['history']).to_csv(history_temp, index=False)
        history_temp.replace(folder/'history.csv')
        write(folder/'validation_metrics_recovered.json', dict(
            **values, epoch=state['best_epoch'], test_opened=False,
            weights_sha256=digest(folder/state['best_weights'])))
    else:
        raise ValueError('Tác vụ không hợp lệ')
'''
_fix_patched = ast.unparse(ast.fix_missing_locations(_fix_tree)) + "\n" + _fix_child_tail
compile(_fix_patched, "cnn_worker_isolated", "exec")
_FIX_WORKER = _FIX_ROOT / "code/cnn_worker_isolated.py"
_FIX_WORKER.write_text(_fix_patched, encoding="utf-8")
_fix_write(_FIX_ROOT / "cnn_memory_recovery.json", dict(
    patch=_FIX_TAG, original_sha256=_fix_hash(_fix_original),
    patched_sha256=_fix_hash(_FIX_WORKER), batch_size=64,
    settings_sha256=_fix_hash(_FIX_ROOT / "comparison_config.json"),
    algorithm_changed=False, test_opened=False,
    changes=["one process per trial/seed", "load resume without extra build", "validation history fields", "retain best checkpoints"]))


def _fix_trial_state(folder, method, seed, lr):
    expected = dict(method=method, seed=int(seed), learning_rate=float(lr),
                    data_sha256=_FIX_READY["train_val_sha256"], settings=_FIX_SETTINGS)
    if (folder / "identity.json").exists() and _fix_read(folder / "identity.json") != expected:
        raise RuntimeError("Khác cấu hình cũ: " + str(folder))
    state = _fix_read(folder / "state.json") if (folder / "state.json").exists() else None
    if state:
        if not (folder / "identity.json").exists():
            raise RuntimeError("Có state nhưng thiếu identity: " + str(folder))
        for key in (["best_weights"] if state["complete"] else ["best_weights", "latest_model"]):
            if not state.get(key) or not (folder / state[key]).is_file():
                raise RuntimeError("Thiếu checkpoint đã ghi trong state, không tự train lại: " + str(folder / str(state.get(key))))
    elif folder.exists() and any(folder.glob("*.keras")):
        raise RuntimeError("Có checkpoint nhưng thiếu state; cần kiểm tra, không ghi đè: " + str(folder))
    return state


def _fix_child(task, log):
    env = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1",
           "KLTN_CNN_ISOLATED_TASK": json.dumps(task)}
    cmd = [sys.executable, "-u", str(_FIX_WORKER), "train", "--run", str(_FIX_ROOT), "--method", task["method"]]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace", env=env)
    try:
        for line in p.stdout:
            log.write(line); log.flush(); print(line, end="", flush=True)
        status = p.wait()
    except BaseException:
        p.terminate()
        try:
            p.wait(timeout=20)
        except subprocess.TimeoutExpired:
            p.kill(); p.wait()
        raise
    finally:
        p.stdout.close()
    if status:
        raise RuntimeError("Tiến trình riêng vẫn lỗi. Đã dừng, không giảm batch/tự train lại. Gửi 30 dòng log cuối.")


def _fix_train_method(method):
    if method not in _FIX_SETTINGS["methods"]:
        raise ValueError("Phương pháp không nằm trong cấu hình đã khóa")
    if _fix_read(_FIX_ROOT / "comparison_config.json") != _FIX_SETTINGS:
        raise RuntimeError("Cấu hình đã đổi sau khi cài cell khôi phục")
    base = _FIX_ROOT / "optimized_cnn_baselines"
    method_dir = base / method
    method_dir.mkdir(parents=True, exist_ok=True)
    first_seed = int(_FIX_SETTINGS["seeds"][0])
    logpath = _FIX_ROOT / ("train_isolated_" + method + ".log")
    with logpath.open("a", encoding="utf-8") as log:
        def trial(folder, seed, lr):
            state = _fix_trial_state(folder, method, seed, lr)
            if state and state["complete"]:
                print("Giữ kết quả đã xong:", folder.relative_to(_FIX_ROOT), flush=True)
            else:
                print("Tiếp tục sau epoch" if state else "Lượt chưa chạy:",
                      state["epoch"] if state else "", folder.relative_to(_FIX_ROOT), flush=True)
                _fix_child(dict(action="trial", folder=str(folder.relative_to(_FIX_ROOT)),
                                method=method, seed=seed, lr=lr), log)
                state = _fix_trial_state(folder, method, seed, lr)
            if not state or not state["complete"]:
                raise RuntimeError("Lượt chưa hoàn tất: " + str(folder))
            return state

        trials = []
        for i, lr in enumerate(_FIX_SETTINGS["learning_rates"]):
            folder = method_dir / "tuning" / f"lr_{i}"
            state = trial(folder, first_seed, float(lr))
            trials.append(dict(lr=float(lr), score=state["best_score"], folder=str(folder.relative_to(_FIX_ROOT))))
        winner = max(trials, key=lambda t: t["score"])
        _fix_write(method_dir / "tuning.json", dict(trials=trials, selected=winner, criterion="validation_macro_f1"))
        for seed in _FIX_SETTINGS["seeds"]:
            folder = _FIX_ROOT / winner["folder"] if int(seed) == first_seed else method_dir / f"train_seed_{seed}"
            state = trial(folder, int(seed), winner["lr"])
            selected = method_dir / f"seed_{seed}"
            selected.mkdir(exist_ok=True)
            _fix_write(selected / "selected.json", dict(
                seed=int(seed), method=method, learning_rate=winner["lr"],
                best_epoch=state["best_epoch"], best_val_macro_f1=state["best_score"], complete=True,
                classifier_weights=str((folder / state["best_weights"]).relative_to(_FIX_ROOT)),
                history=str((folder / "history.csv").relative_to(_FIX_ROOT)),
                train_val_sha256=_FIX_READY["train_val_sha256"], test_sha256=_FIX_READY["test_sha256"]))

        # Bổ sung độ đo còn thiếu ở checkpoint đã chọn, kể cả CNN thường đã xong.
        # Chỉ dự đoán validation; tuyệt đối không train lại các lượt hoàn thành.
        for selected_path in sorted(base.glob("*/seed_*/selected.json")):
            selected = _fix_read(selected_path)
            if not selected["complete"]:
                continue
            folder = (_FIX_ROOT / selected["history"]).parent
            state = _fix_trial_state(folder, selected["method"], selected["seed"], selected["learning_rate"])
            best_row = next(r for r in state["history"] if r["epoch"] == state["best_epoch"])
            if any(not isinstance(best_row.get(k), (int, float)) or not math.isfinite(best_row[k])
                   for k in ["val_gm", "val_kappa"]):
                _fix_child(dict(action="validation_history", folder=str(folder.relative_to(_FIX_ROOT)),
                                method=selected["method"], seed=selected["seed"], lr=selected["learning_rate"]), log)
    print("HOÀN THÀNH:", method, "| batch=64 |", _FIX_SETTINGS["seeds"], flush=True)


# Những cell huấn luyện cũ tự gọi cách chạy mới. GAN/smoke/test không bị đổi.
if not getattr(run_worker, "_cnn_memory_fix", False):
    _run_worker_before_memory_fix = run_worker


def run_worker(kind, stage, method=None, child=None):
    if kind != "cnn" or stage != "train":
        return _run_worker_before_memory_fix(kind, stage, method=method, child=child)
    try:
        return _fix_train_method(method)
    finally:
        print("Đã cập nhật ZIP:", zip_run(), flush=True)


run_worker._cnn_memory_fix = True
EVALUATE_TEST = False
print("Thư mục giữ nguyên:", _FIX_ROOT)
print("Batch CNN:", _FIX_SETTINGS["batch_size"], "| Seeds:", _FIX_SETTINGS["seeds"])
try:
    _gpu_status = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,memory.free", "--format=csv"],
        capture_output=True, text=True, timeout=10,
    )
    if _gpu_status.returncode == 0:
        print(_gpu_status.stdout)
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("Không đọc được nvidia-smi; worker vẫn sẽ kiểm tra GPU trước huấn luyện.")
for _path in sorted((_FIX_ROOT / "optimized_cnn_baselines").glob("*/**/state.json")):
    _state = _fix_read(_path)
    print(_path.parent.relative_to(_FIX_ROOT), "| epoch", _state["epoch"], "| xong:", _state["complete"])
print("SẴN SÀNG — chạy lại cell CNN + ROS, rồi các cell phía sau. Không Run All từ đầu.")
print("Nếu còn lỗi GPU, dừng và gửi log; chưa có cơ sở bắt buộc giảm batch.")


In [ ]:
if RUN_TRAINING: run_worker('cnn','train',method='cnn3d_random_oversampling')
plot_cnn_training('cnn3d_random_oversampling')


In [ ]:
# TIẾP TỤC CNN + ROS TỪ CHECKPOINT seed 43 lr 0.001 epoch 21

if RUN_TRAINING:
    run_worker(
        'cnn',
        'train',
        method='cnn3d_random_oversampling'
    )

plot_cnn_training('cnn3d_random_oversampling')

## 14. CNN + trọng số lớp

Không tạo hoặc lặp ảnh. Trọng số lớp = số mẫu train / (16 × số mẫu của lớp). Lớp ít mẫu có ảnh hưởng lớn hơn lên hàm mất mát. Mô hình tốt nhất được chọn bằng validation, không bằng test.

In [ ]:
if RUN_TRAINING: run_worker('cnn','train',method='cnn3d_class_weighting')
plot_cnn_training('cnn3d_class_weighting')


## 15. Huấn luyện 3D-HyperGAMO

Giữ mạng G/C/D, cơ chế tổ hợp lồi và trình tự đối kháng của nguồn tác giả. 8.000 là số lượt cập nhật bộ phân lớp, tương ứng 4.000 vòng, không phải 8.000 epoch. Nếu chạm giới hạn phút, lưu checkpoint và đánh dấu chưa hoàn tất; không đưa kết quả dở dang vào bảng cuối.

In [ ]:
if RUN_TRAINING:
    for seed,child in GAN_RUNS.items(): run_worker('gan','train',child=child)
for seed,child in GAN_RUNS.items():
    path=child/'training_state.json'
    print('Seed',seed,read_json(path) if path.exists() else 'Chưa huấn luyện')


### 15.1. Quá trình học của GAN

Xem riêng loss của bộ phân lớp, bộ phân biệt và bộ sinh; không suy ra chất lượng ảnh chỉ từ một đường loss.

In [ ]:
for seed,child in GAN_RUNS.items():
    p=child/'training_history.csv'
    if not p.exists(): continue
    h=pd.read_csv(p); display(h.tail())
    fig,ax=plt.subplots(1,2,figsize=(13,4))
    for k in ['OA','AA','Macro_F1','GM']:
        if k in h: ax[0].plot(h.step,h[k],label=k)
    for k in ['C_loss','D_loss','G_vs_C_loss','G_vs_D_loss']:
        if k in h: ax[1].plot(h.step,h[k],label=k)
    ax[0].set_title('Validation'); ax[1].set_title('Các hàm mất mát đối kháng')
    for a in ax: a.legend(); a.set_xlabel('Lượt cập nhật bộ phân lớp'); a.grid(alpha=.2)
    fig.suptitle(f'3D-HyperGAMO — seed {seed}'); fig.tight_layout(); show_fig(f'15_gan_learning_{seed}',fig)


## 16. Cách bốn phương pháp xử lý dữ liệu train

Biểu đồ số mẫu và trọng số không thay thế chất lượng phân loại. Số mẫu GAN là **lượt mẫu sinh đã đưa vào học**, không phải số ảnh giả độc nhất.

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4)); ids=count_table.class_id.to_numpy(); n=count_table.train.to_numpy()
ax[0].bar(ids-.2,n,.4,label='CNN thường / trọng số lớp'); ax[0].bar(ids+.2,np.full(16,n.max()),.4,label='ROS')
ax[0].set_title('Số mẫu mỗi epoch'); ax[0].legend(fontsize=8)
ax[1].bar(ids,n.sum()/(16*n)); ax[1].set_title('Trọng số lớp của CNN')
p=GAN_RUNS[SEEDS[0]]/'online_generated_counts.json'
if p.exists():
    g=read_json(p); ax[2].bar(g['class_ids'],g['generated_presentations']); ax[2].set_title('Lượt mẫu GAN đã đưa cho bộ phân lớp')
else: ax[2].text(.1,.5,'Chưa có thống kê GAN',transform=ax[2].transAxes)
for a in ax: a.set_xticks(ids); a.set_xlabel('Mã lớp')
fig.tight_layout(); show_fig('16_balance_treatments',fig)


## 17. Nhìn trực tiếp ảnh thật và mẫu GAN

Mẫu GAN có nhãn điều kiện, không có “ảnh thật đích” ghép cặp. Ảnh thật bên cạnh là **mẫu train gần nhất**, không phải ảnh mà GAN bắt buộc tái tạo. Hiển thị màu giả bằng ba dải được tái dựng từ PCA, chung thang màu; không phải ảnh RGB đo mới.

In [ ]:
for seed,child in GAN_RUNS.items():
    state=child/'training_state.json'
    if RUN_TRAINING and state.exists() and read_json(state)['complete']:
        run_worker('gan','samples',child=child)
EXAMPLES=GAN_RUNS[SEEDS[0]]/'generated_examples.npz'
def inverse_patch(x):
    shape=x.shape[:-2]
    return pca.inverse_transform(scaler.inverse_transform(x[...,0].reshape(-1,40))).reshape(*shape,200)
if EXAMPLES.exists():
    with np.load(EXAMPLES) as d:
        fake=d['fake']; real=d['nearest_real']; fake_ids=d['original_class_ids']
    chosen=np.unique(fake_ids)[:4]
    fig,ax=plt.subplots(len(chosen),5,figsize=(14,3*len(chosen)),squeeze=False)
    titles=['Train tham chiếu','CNN thường: không sinh','ROS: lặp ảnh thật','Trọng số lớp: không sinh','3D-HyperGAMO: mẫu mới']
    for row,cid in enumerate(chosen):
        j=np.flatnonzero(fake_ids==cid)[0]; r=inverse_patch(real[j]); f=inverse_patch(fake[j])
        rr=r[:,:,[30,20,10]]; ff=f[:,:,[30,20,10]]
        both=np.concatenate([rr.reshape(-1,3),ff.reshape(-1,3)])
        lo=np.percentile(both,2,axis=0); hi=np.percentile(both,98,axis=0)
        rr=np.clip((rr-lo)/(hi-lo+1e-8),0,1); ff=np.clip((ff-lo)/(hi-lo+1e-8),0,1)
        for col,img in enumerate([rr,rr,rr,rr,ff]):
            ax[row,col].imshow(img); ax[row,col].axis('off')
            ax[row,col].set_title(titles[col] if row==0 else f'Lớp {cid}')
        ax[row,0].set_title(f'Lớp {cid}: '+CLASS_NAMES[cid-1])
    fig.suptitle('Minh họa cách dùng dữ liệu — ảnh CNN/ROS là cùng mẫu thật tham chiếu, không phải ảnh sinh',y=1.01)
    fig.tight_layout(); show_fig('17_real_and_generated',fig)
else: print('Chưa có mẫu GAN; không tạo ảnh minh họa thay kết quả.')


### 17.1. So sánh phổ và kiểm tra mức độ lặp mẫu

Phổ dưới đây được tái dựng từ PCA. Khoảng cách nhỏ với mẫu train có thể cho thấy mẫu sinh gần mẫu có sẵn; chưa đủ để kết luận mô hình suy biến.

In [ ]:
if EXAMPLES.exists():
    chosen=np.unique(fake_ids)[:4]; fig,ax=plt.subplots(2,2,figsize=(13,7))
    center=CFG['patch_size']//2
    for a,cid in zip(ax.flat,chosen):
        j=np.flatnonzero(fake_ids==cid)[0]
        a.plot(inverse_patch(real[j])[center,center],label='Train gần nhất (tái dựng)')
        a.plot(inverse_patch(fake[j])[center,center],'--',label='GAN (tái dựng)')
        a.set_title(f'Lớp {cid}: {CLASS_NAMES[cid-1]}'); a.set_xlabel('Chỉ số dải phổ'); a.legend(fontsize=8); a.grid(alpha=.2)
    fig.tight_layout(); show_fig('17_spectra',fig)
    diagnostic=GAN_RUNS[SEEDS[0]]/'generation_diagnostics.csv'
    if diagnostic.exists(): display(pd.read_csv(diagnostic))


## 18. So sánh validation chính thức

Bảng này dùng để quyết định bước tối ưu/cải tiến GAN tiếp theo. **Không cần mở test.**

- CNN: lấy đúng epoch đã được chọn bằng validation Macro-F1.
- 3D-HyperGAMO: lấy đúng checkpoint được chọn bằng validation Macro-F1.
- Báo cáo từng seed và trung bình ± độ lệch chuẩn qua 3 seed.


In [ ]:
# ============================================================
# SỬA val_gm / val_kappa BỊ THIẾU
# KHÔNG TRAIN LẠI MODEL
# ============================================================

from pathlib import Path
import json
import os
import sys
import subprocess
import pandas as pd

# Run hiện tại
ROOT = Path("/kaggle/working/comparison_4methods_20fffe805617")

assert ROOT.exists(), f"Không thấy run: {ROOT}"

BASE = ROOT / "optimized_cnn_baselines"
assert BASE.exists(), f"Không thấy CNN baseline: {BASE}"

# ------------------------------------------------------------
# 1. Tìm patched CNN worker có hỗ trợ validation_history
# ------------------------------------------------------------
workers = []

for p in ROOT.rglob("*.py"):
    try:
        text = p.read_text(encoding="utf-8")
    except Exception:
        continue

    if "validation_history" in text and "KLTN_CNN_ISOLATED_TASK" in text:
        workers.append(p)

print("Worker tìm được:")
for p in workers:
    print(" -", p)

if len(workers) != 1:
    raise RuntimeError(
        f"Cần đúng 1 patched worker, nhưng tìm được {len(workers)}. "
        "Dừng lại, không train hay sửa file nào."
    )

WORKER = workers[0]

# ------------------------------------------------------------
# 2. Kiểm tra selected CNN nào thiếu val_gm / val_kappa
# ------------------------------------------------------------
jobs = []

for selected_path in sorted(BASE.glob("*/seed_*/selected.json")):

    sel = json.loads(
        selected_path.read_text(encoding="utf-8")
    )

    if not sel.get("complete", False):
        print("BỎ QUA chưa complete:", selected_path)
        continue

    history_path = ROOT / sel["history"]

    assert history_path.is_file(), \
        f"Thiếu history: {history_path}"

    h = pd.read_csv(history_path)

    missing = [
        col for col in ["val_gm", "val_kappa"]
        if col not in h.columns
    ]

    # Cũng sửa nếu cột tồn tại nhưng checkpoint được chọn bị NaN
    best_epoch = int(sel["best_epoch"])

    selected_row = h.loc[
        h["epoch"].astype(int) == best_epoch
    ]

    if selected_row.empty:
        raise RuntimeError(
            f"Không thấy best_epoch={best_epoch} trong {history_path}"
        )

    bad_value = False

    for col in ["val_gm", "val_kappa"]:
        if col in h.columns:
            value = selected_row.iloc[0][col]
            if pd.isna(value):
                bad_value = True

    if missing or bad_value:
        folder = history_path.parent

        jobs.append({
            "action": "validation_history",
            "folder": str(folder.relative_to(ROOT)),
            "method": sel["method"],
            "seed": int(sel["seed"]),
            "lr": float(sel["learning_rate"]),
        })

        print(
            "CẦN SỬA:",
            sel["method"],
            "seed", sel["seed"],
            "| thiếu:", missing,
            "| best epoch:", best_epoch
        )

    else:
        print(
            "ĐÃ ĐỦ:",
            sel["method"],
            "seed", sel["seed"]
        )

print("\nSố checkpoint cần bổ sung metric:", len(jobs))

# ------------------------------------------------------------
# 3. Chỉ load checkpoint + dự đoán validation
#    KHÔNG TRAIN
# ------------------------------------------------------------
for task in jobs:

    print("\n======================================")
    print(
        "Đang bổ sung:",
        task["method"],
        "seed", task["seed"]
    )
    print("======================================")

    env = {
        **os.environ,
        "PYTHONIOENCODING": "utf-8",
        "PYTHONUNBUFFERED": "1",
        "KLTN_CNN_ISOLATED_TASK": json.dumps(task),
    }

    cmd = [
        sys.executable,
        "-u",
        str(WORKER),
        "train",
        "--run",
        str(ROOT),
        "--method",
        task["method"],
    ]

    result = subprocess.run(
        cmd,
        env=env,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Lỗi khi bổ sung validation metric cho {task}"
        )

# ------------------------------------------------------------
# 4. KIỂM TRA LẠI
# ------------------------------------------------------------
print("\n========== KIỂM TRA SAU SỬA ==========")

all_ok = True

for selected_path in sorted(BASE.glob("*/seed_*/selected.json")):

    sel = json.loads(
        selected_path.read_text(encoding="utf-8")
    )

    if not sel.get("complete", False):
        continue

    h = pd.read_csv(ROOT / sel["history"])

    ok = (
        "val_gm" in h.columns
        and "val_kappa" in h.columns
    )

    print(
        sel["method"],
        "seed", sel["seed"],
        "→",
        "OK" if ok else "THIẾU"
    )

    all_ok &= ok

assert all_ok, "Vẫn còn history thiếu metric."

print("\n✅ ĐÃ SỬA XONG val_gm / val_kappa")
print("✅ KHÔNG TRAIN LẠI CNN")
print("✅ KHÔNG TRAIN LẠI GAN")
print("Giờ chạy lại cell tổng hợp kết quả bị KeyError.")

In [ ]:
validation_rows = []

for method in METHODS:
    for seed in SEEDS:
        if method == 'hypergamo':
            child = GAN_RUNS[seed]
            state_path = child / 'training_state.json'
            selection_path = child / 'selection.json'
            if not state_path.exists() or not selection_path.exists():
                continue

            state = read_json(state_path)
            if not state.get('complete'):
                continue

            sel = read_json(selection_path)
            validation_rows.append(dict(
                Method=NAMES[method],
                Seed=seed,
                OA=float(sel['OA']),
                AA=float(sel['AA']),
                Macro_F1=float(sel['Macro_F1']),
                G_mean=float(sel['GM']),
                Kappa=float(sel['Kappa']),
                Selected=int(sel['step']),
            ))

        else:
            folder = selected_dir(method, seed)
            selected_path = folder / 'selected.json'
            if not selected_path.exists():
                continue

            sel = read_json(selected_path)
            if not sel.get('complete'):
                continue

            history = pd.read_csv(RUN / sel['history'])
            best = history.loc[
                history['epoch'] == int(sel['best_epoch'])
            ]

            if len(best) != 1:
                raise RuntimeError(
                    f'Không tìm đúng best epoch của {method}, seed {seed}'
                )

            row = best.iloc[0]
            validation_rows.append(dict(
                Method=NAMES[method],
                Seed=seed,
                OA=float(row['val_oa']),
                AA=float(row['val_aa']),
                Macro_F1=float(row['val_macro_f1']),
                G_mean=float(row['val_gm']),
                Kappa=float(row['val_kappa']),
                Selected=int(sel['best_epoch']),
            ))

validation_results = pd.DataFrame(validation_rows)
VAL_COMPLETE = (
    len(validation_results) == len(METHODS) * len(SEEDS)
)

if len(validation_results):
    validation_results.to_csv(
        RUN / 'validation_comparison_by_seed.csv',
        index=False,
    )
    display(validation_results)

if VAL_COMPLETE:
    metric_cols = [
        'OA', 'AA', 'Macro_F1', 'G_mean', 'Kappa'
    ]

    val_means = (
        validation_results
        .groupby('Method', sort=False)[metric_cols]
        .mean()
    )

    val_stds = (
        validation_results
        .groupby('Method', sort=False)[metric_cols]
        .std(ddof=1)
    )

    val_means.to_csv(
        RUN / 'validation_comparison_mean.csv'
    )
    val_stds.to_csv(
        RUN / 'validation_comparison_std.csv'
    )

    val_shown = val_means.copy().astype(object)

    for col in metric_cols:
        factor = 1 if col == 'Kappa' else 100
        val_shown[col] = [
            f'{val_means.loc[m, col] * factor:.2f} ± '
            f'{val_stds.loc[m, col] * factor:.2f}'
            for m in val_means.index
        ]

    display(Markdown(
        '### Validation — trung bình ± độ lệch chuẩn qua 3 seed'
    ))
    display(val_shown)

else:
    print(
        'Validation chưa đủ:',
        len(validation_results),
        '/',
        len(METHODS) * len(SEEDS),
        'run hoàn tất.'
    )


## 18. Cổng kiểm tra trước đánh giá

Chỉ mở test khi cả bốn phương pháp hoàn tất trên toàn bộ seed đã khai báo. Nếu chưa đủ, notebook vẫn lưu kết quả hiện có và ghi rõ cần tiếp tục; không công bố bảng thiếu thành phần.

In [ ]:
status=[]
for method in METHODS:
    for seed in SEEDS:
        p=selected_dir(method,seed)/('training_state.json' if method=='hypergamo' else 'selected.json')
        status.append(dict(Method=NAMES[method],Seed=seed,Complete=p.exists() and bool(read_json(p).get('complete'))))
status=pd.DataFrame(status); display(status)
ALL_COMPLETE=bool(status.Complete.all())
write_json(RUN/'completion_status.json',{'all_training_complete':ALL_COMPLETE,'validation_comparison_complete':bool(VAL_COMPLETE),'rows':status.to_dict('records')})
if EVALUATE_TEST and ALL_COMPLETE:
    run_worker('cnn','evaluate')
    for child in GAN_RUNS.values(): run_worker('gan','evaluate',child=child)
else: print('TEST ĐANG KHÓA: checkpoint/validation vẫn được giữ; chỉ bật EVALUATE_TEST=True sau khi khóa cấu hình cuối.')
HAVE_RESULTS=ALL_COMPLETE and all((selected_dir(m,s)/'test_predictions.npz').exists() for m in METHODS for s in SEEDS)
print('Đủ dữ liệu so sánh:',HAVE_RESULTS)


## 19. Bảng kết quả chung

Đánh giá lại từ dự đoán đã lưu trên **đúng cùng tọa độ và nhãn test**, không ghép các con số của phiên bản khác. OA = tỷ lệ đúng; AA = trung bình recall các lớp; Macro-F1 = trung bình F1 các lớp; G-mean giảm mạnh khi có lớp bị bỏ sót; Kappa đo mức đồng thuận có hiệu chỉnh. Các điểm số lưu dạng 0–1, bảng hiển thị phần trăm trừ Kappa.

In [ ]:
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,cohen_kappa_score,confusion_matrix
PREDICTIONS={}; rows=[]; per_class=[]
if HAVE_RESULTS:
    reference_coords=None; reference_y=None
    for method in METHODS:
        for seed in SEEDS:
            with np.load(selected_dir(method,seed)/'test_predictions.npz') as d:
                pred={k:d[k].copy() for k in ['coords','true_class_ids','predicted_class_ids']}
            if reference_coords is None: reference_coords=pred['coords']; reference_y=pred['true_class_ids']
            np.testing.assert_array_equal(pred['coords'],reference_coords)
            np.testing.assert_array_equal(pred['true_class_ids'],reference_y)
            np.testing.assert_array_equal(pred['coords'],coords_by_split[2])
            yhat=pred['predicted_class_ids']; ytrue=pred['true_class_ids']
            precision,recall,f1,support=precision_recall_fscore_support(ytrue,yhat,labels=np.arange(1,17),zero_division=0)
            minority=count_table.train.to_numpy()<=np.median(count_table.train)
            rows.append(dict(Method=NAMES[method],Seed=seed,OA=accuracy_score(ytrue,yhat),AA=recall.mean(),
                Macro_F1=f1.mean(),G_mean=np.exp(np.log(recall).mean()) if (recall>0).all() else 0.,
                Kappa=cohen_kappa_score(ytrue,yhat),Minority_Recall=recall[minority].mean()))
            for k in range(16): per_class.append(dict(Method=NAMES[method],Seed=seed,Class_ID=k+1,
                Class_Name=CLASS_NAMES[k],Precision=precision[k],Recall=recall[k],F1=f1[k],Test_Support=int(support[k])))
            PREDICTIONS[(method,seed)]=pred
    results=pd.DataFrame(rows); classes=pd.DataFrame(per_class)
    results.to_csv(RUN/'comparison_by_seed.csv',index=False); classes.to_csv(RUN/'comparison_per_class.csv',index=False)
    means=results.groupby('Method',sort=False).mean(numeric_only=True).drop(columns='Seed')
    stds=results.groupby('Method',sort=False).std(numeric_only=True,ddof=1).drop(columns='Seed')
    means.to_csv(RUN/'comparison_mean.csv'); stds.to_csv(RUN/'comparison_std.csv')
    shown=means.copy()
    for col in shown:
        factor=1 if col=='Kappa' else 100
        shown[col]=[f'{means.loc[m,col]*factor:.2f}'+(f' ± {stds.loc[m,col]*factor:.2f}' if len(SEEDS)>1 else '') for m in means.index]
    display(shown)
    print('Lớp thiểu số theo số TRAIN ≤ trung vị:',count_table.loc[minority,'class_id'].tolist())
    print('Một seed: chưa đánh giá độ ổn định.' if len(SEEDS)==1 else 'Bảng: trung bình ± độ lệch chuẩn giữa các seed.')
else: print('Chưa đủ kết quả; không điền số minh họa.')


### 19.1. Nhìn nhanh điểm mạnh và điểm yếu

Không chỉ nhìn OA: xem Macro-F1 và recall lớp ít mẫu để tránh bị số lượng lớp lớn che khuất.

In [ ]:
if HAVE_RESULTS:
    cols=['OA','AA','Macro_F1','G_mean','Minority_Recall']
    fig,ax=plt.subplots(figsize=(12,5))
    (means[cols]*100).plot.bar(ax=ax,rot=0); ax.set_ylim(0,105)
    ax.set_ylabel('Điểm (%)'); ax.set_xlabel(''); ax.legend(loc='upper center',ncol=3); ax.grid(axis='y',alpha=.2)
    fig.tight_layout(); show_fig('19_metric_comparison',fig)
    tab=classes.groupby(['Method','Class_ID']).Recall.mean().unstack(0).reindex(columns=[NAMES[m] for m in METHODS])
    fig,ax=plt.subplots(figsize=(9,9)); im=ax.imshow(tab.values*100,vmin=0,vmax=100,cmap='YlGnBu',aspect='auto')
    ax.set_xticks(range(4),tab.columns,rotation=15); ax.set_yticks(range(16),[f'{i+1}. {s}' for i,s in enumerate(CLASS_NAMES)])
    for r in range(16):
        for c in range(4): ax.text(c,r,f'{tab.iloc[r,c]*100:.1f}',ha='center',va='center',color='white' if tab.iloc[r,c]>.6 else 'black')
    fig.colorbar(im,ax=ax,label='Recall (%)'); ax.set_title('Mỗi lớp được nhận ra bao nhiêu phần trăm?')
    fig.tight_layout(); show_fig('19_class_recall',fig)


## 20. So sánh bản đồ phân loại của cả bốn mô hình

Chỉ tô **tâm test**, không trộn điểm train vào bản đồ để làm ảnh đẹp hơn. Các mô hình dùng cùng màu lớp, cùng vị trí. Hình dùng seed đầu tiên đã khai báo, không chọn seed có điểm test đẹp nhất.

In [ ]:
if HAVE_RESULTS:
    colors=['#000000']+[plt.get_cmap('tab20')(i) for i in range(16)]
    cmap=ListedColormap(colors); norm=BoundaryNorm(np.arange(-.5,17.5),17)
    maps={}; true_map=np.zeros(gt.shape,dtype=int)
    rc=reference_coords; true_map[rc[:,0],rc[:,1]]=reference_y
    for m in METHODS:
        a=np.zeros(gt.shape,dtype=int); a[rc[:,0],rc[:,1]]=PREDICTIONS[(m,SEEDS[0])]['predicted_class_ids']; maps[m]=a
    fig,axs=plt.subplots(1,5,figsize=(20,5))
    for ax,(name,arr) in zip(axs,[('Nhãn test',true_map)]+[(NAMES[m],maps[m]) for m in METHODS]):
        ax.imshow(arr,cmap=cmap,norm=norm,interpolation='nearest'); ax.set_title(name); ax.axis('off')
    fig.tight_layout(); show_fig('20_test_maps',fig)
    from matplotlib.patches import Patch
    fig,ax=plt.subplots(figsize=(14,2.5)); ax.axis('off')
    ax.legend(handles=[Patch(color=colors[i],label=f'{i}: {CLASS_NAMES[i-1]}') for i in range(1,17)],ncol=4,loc='center',frameon=False)
    show_fig('20_map_legend',fig)


### 20.1. Bản đồ lỗi — màu đỏ là phân loại sai

Cùng vị trí nhưng mô hình nào sai ở đâu? Đây là cách đối chiếu trực tiếp thay vì chỉ so sánh một con số.

In [ ]:
if HAVE_RESULTS:
    fig,axs=plt.subplots(1,4,figsize=(17,5))
    for ax,m in zip(axs,METHODS):
        correct=PREDICTIONS[(m,SEEDS[0])]['predicted_class_ids']==reference_y
        a=np.zeros(gt.shape,dtype=int); a[rc[:,0],rc[:,1]]=np.where(correct,1,2)
        ax.imshow(a,cmap=ListedColormap(['#161616','#80b1d3','#e31a1c']),vmin=0,vmax=2)
        ax.set_title(f'{NAMES[m]} — Sai {(~correct).sum()} / {len(correct)}'); ax.axis('off')
    fig.suptitle('Đen: không đánh giá; xanh: đúng; đỏ: sai'); fig.tight_layout(); show_fig('20_error_maps',fig)


### 20.2. Ma trận nhầm lẫn

Hàng là lớp thật, cột là lớp dự đoán; chuẩn hóa từng hàng giúp thấy lớp nhỏ có bị nhầm hay không.

In [ ]:
if HAVE_RESULTS:
    fig,axs=plt.subplots(2,2,figsize=(13,12))
    for ax,m in zip(axs.flat,METHODS):
        cm=confusion_matrix(reference_y,PREDICTIONS[(m,SEEDS[0])]['predicted_class_ids'],labels=range(1,17))
        cm=cm/np.maximum(cm.sum(1,keepdims=True),1)*100
        im=ax.imshow(cm,vmin=0,vmax=100,cmap='Blues')
        ax.set_xticks(range(16),range(1,17)); ax.set_yticks(range(16),range(1,17)); ax.set_title(NAMES[m])
        ax.set_xlabel('Lớp dự đoán'); ax.set_ylabel('Lớp thật')
        for r in range(16):
            for c in range(16):
                if cm[r,c]>=5: ax.text(c,r,f'{cm[r,c]:.0f}',ha='center',va='center',fontsize=7,color='white' if cm[r,c]>50 else 'black')
    fig.tight_layout(); show_fig('20_confusion_matrices',fig)


## 21. Đọc kết quả mà không kết luận quá mức

- Bảng điểm trả lời: trên **cùng giao thức DIP hiện tại**, quy trình nào có điểm cao hơn, lớp nào được cải thiện hoặc bị giảm.
- Ảnh mẫu và phổ giúp kiểm tra mẫu GAN có khác và có gần dữ liệu train hay không. Không dùng ảnh màu giả để khẳng định độ đúng vật lý của 200 dải phổ.
- Bản đồ lỗi cho biết sự khác nhau giữa các mô hình tại cùng điểm test.
- Không kết luận “đã hết rò rỉ”, “GAN tối ưu nhất”, hoặc “GAN luôn tốt hơn” từ một bộ dữ liệu/một seed. Không chọn lại cấu hình bằng test sau khi xem bảng.
- Bước cải tiến chỉ chọn sau khi xem các kết quả này; notebook này **chưa cài phương pháp đề xuất mới**.

In [ ]:
if HAVE_RESULTS:
    best=means['Macro_F1'].idxmax()
    display(Markdown(f'**Macro-F1 trung bình cao nhất trong lần chạy: {best} ({means.loc[best,"Macro_F1"]*100:.2f}%).** Đây là mô tả kết quả, chưa phải kết luận khác biệt có ý nghĩa thống kê.'))
    print('Chênh lệch Macro-F1 so với CNN thường (điểm phần trăm):')
    display(((means.Macro_F1-means.loc[NAMES['cnn3d_real_only'],'Macro_F1'])*100).round(2))


## 22. Lưu toàn bộ — thao tác cuối cùng

ZIP chứa cấu hình, mã nguồn, phép biến đổi, phân chia dữ liệu, checkpoint, dự đoán từng điểm, CSV và tất cả hình. `BAO_CAO_HINH.html` trình bày hình và bảng ngay cả khi không mở notebook.

**Chờ phiên bản Save & Run All hoàn tất → Output → tải ZIP.** Kiểm tra ZIP có `comparison_by_seed.csv`, `comparison_per_class.csv`, thư mục `figures` và checkpoint của cả CNN/GAN. Nếu chưa hoàn tất, ZIP vẫn giữ tiến độ để tiếp tục, không phải kết quả cuối.

In [ ]:
import html
summary_html='<h1>Benchmark chính thức bốn phương pháp — Indian Pines</h1><p>DIP 16 lớp; PCA train-only; vùng ảnh có chồng lấn.</p>'
summary_html+=f'<p>Huấn luyện hoàn tất: {ALL_COMPLETE}; validation đủ: {VAL_COMPLETE}; test đủ: {HAVE_RESULTS}; seed: {SEEDS}</p>'
if VAL_COMPLETE: summary_html+='<h2>Validation benchmark</h2>'+val_shown.to_html()
if HAVE_RESULTS: summary_html+=shown.to_html()
for image_path in sorted([*RUN.glob('*.png'),*FIGDIR.glob('*.png')]):
    rel=image_path.relative_to(RUN).as_posix()
    summary_html+=f'<h2>{html.escape(image_path.stem)}</h2><img style="max-width:100%" src="{rel}">'
(RUN/'BAO_CAO_HINH.html').write_text('<!doctype html><meta charset="utf-8"><body style="font-family:Arial;max-width:1200px;margin:30px auto">'+summary_html+'</body>',encoding='utf-8')
write_json(RUN/'final_status.json',{'training_complete':ALL_COMPLETE,'comparison_complete':HAVE_RESULTS,
    'seeds':SEEDS,'protocol':CFG['protocol'],'validation_complete':bool(VAL_COMPLETE),
    'spatial_overlap_present':True})
archive=zip_run()
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    print('ZIP hợp lệ:',len(z.namelist()),'tệp')
print('ĐÃ LƯU:',archive)
print('Dung lượng:',round(archive.stat().st_size/1024**2,1),'MiB')
print('HOÀN TẤT SO SÁNH' if HAVE_RESULTS else 'CHƯA HOÀN TẤT: giữ ZIP và tiếp tục, chưa kết luận kết quả.')
print('Tải từ Output của phiên bản đã lưu. Không đóng phiên trước khi xác nhận Output đã được lưu.')
